# THEMIS-Aの電磁場データについて、full orbit (8 Hz)とparticle burst (512 Hz)、low telemetry (16 Hz)とhigh telemetry (128 Hz)の両方を用いる

# FACの定義の付加

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# THEMIS-Aの電場・磁場データのdownload

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/20:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330'

psp.themis.fgm(trange=time_range, probe='a', level='l2', no_update=True, get_support_data=True)                 # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp', no_update=True, get_support_data=True) # efp: 512 Hz
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efi', no_update=True, get_support_data=True) # eff: 8 Hz
print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
Espin_data_gsm  = pt.data_quants['tha_efs_dot0_gsm']
E8_data_gsm     = pt.data_quants['tha_eff_dot0_gsm']
E512_data_gsm   = pt.data_quants['tha_efp_gsm']
Bspin_data_gsm  = pt.data_quants['tha_fgs_gsm']
B16_data_gsm    = pt.data_quants['tha_fgl_gsm']
B128_data_gsm   = pt.data_quants['tha_fgh_gsm']

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
Espin_data_gsm  = Espin_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
E8_data_gsm     = E8_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
E512_data_gsm   = E512_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
Bspin_data_gsm  = Bspin_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B16_data_gsm    = B16_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B128_data_gsm   = B128_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')

In [ ]:
import xarray as xr
import numpy as np

def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_Espin_data_gsm = xr.Dataset(
    data_vars={
        "Espin_gsm_x": (("time",), Espin_data_gsm[:, 0].values),
        "Espin_gsm_y": (("time",), Espin_data_gsm[:, 1].values),
        "Espin_gsm_z": (("time",), Espin_data_gsm[:, 2].values),
    },
    coords={
        "time": Espin_data_gsm["time"].values,
    },
)
ds_E8_data_gsm = xr.Dataset(
    data_vars={
        "E8_gsm_x": (("time",), E8_data_gsm[:, 0].values),
        "E8_gsm_y": (("time",), E8_data_gsm[:, 1].values),
        "E8_gsm_z": (("time",), E8_data_gsm[:, 2].values),
    },
    coords={
        "time": E8_data_gsm["time"].values,
    },
)
ds_E512_data_gsm = xr.Dataset(
    data_vars={
        "E512_gsm_x": (("time",), E512_data_gsm[:, 0].values),
        "E512_gsm_y": (("time",), E512_data_gsm[:, 1].values),
        "E512_gsm_z": (("time",), E512_data_gsm[:, 2].values),
    },
    coords={
        "time": E512_data_gsm["time"].values,
    },
)
ds_Bspin_data_gsm = xr.Dataset(
    data_vars={
        "Bspin_gsm_x": (("time",), Bspin_data_gsm[:, 0].values),
        "Bspin_gsm_y": (("time",), Bspin_data_gsm[:, 1].values),
        "Bspin_gsm_z": (("time",), Bspin_data_gsm[:, 2].values),
    },
    coords={
        "time": Bspin_data_gsm["time"].values,
    },
)
ds_B16_data_gsm = xr.Dataset(
    data_vars={
        "B16_gsm_x": (("time",), B16_data_gsm[:, 0].values),
        "B16_gsm_y": (("time",), B16_data_gsm[:, 1].values),
        "B16_gsm_z": (("time",), B16_data_gsm[:, 2].values),
    },
    coords={
        "time": B16_data_gsm["time"].values,
    },
)
ds_B128_data_gsm = xr.Dataset(
    data_vars={
        "B128_gsm_x": (("time",), B128_data_gsm[:, 0].values),
        "B128_gsm_y": (("time",), B128_data_gsm[:, 1].values),
        "B128_gsm_z": (("time",), B128_data_gsm[:, 2].values),
    },
    coords={
        "time": B128_data_gsm["time"].values,
    },
)

In [ ]:
def uniq_and_sort_time(ds):
    idx = ds.get_index('time')
    mask = ~idx.duplicated()          # 最初の出現だけ True
    return ds.isel(time=mask).sortby('time')

In [ ]:
ds_Espin_data_gsm   = uniq_and_sort_time(ds_Espin_data_gsm)
ds_E8_data_gsm      = uniq_and_sort_time(ds_E8_data_gsm)
ds_E512_data_gsm    = uniq_and_sort_time(ds_E512_data_gsm)
ds_Bspin_data_gsm   = uniq_and_sort_time(ds_Bspin_data_gsm)
ds_B16_data_gsm     = uniq_and_sort_time(ds_B16_data_gsm)
ds_B128_data_gsm    = uniq_and_sort_time(ds_B128_data_gsm)

In [ ]:
ds_Espin_data_gsm_segs = split_by_gap(ds_Espin_data_gsm, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Espin_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_E8_data_gsm_segs = split_by_gap(ds_E8_data_gsm, gap_thr=np.timedelta64(500, 'ms'))
for ds_ in ds_E8_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_E512_data_gsm_segs = split_by_gap(ds_E512_data_gsm, gap_thr=np.timedelta64(8, 'ms'))
for ds_ in ds_E512_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_Bspin_data_gsm_segs = split_by_gap(ds_Bspin_data_gsm, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Bspin_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_B16_data_gsm_segs = split_by_gap(ds_B16_data_gsm, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B16_data_gsm_segs:
    print(ds_.time)

In [ ]:
ds_B128_data_gsm_segs = split_by_gap(ds_B128_data_gsm, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B128_data_gsm_segs:
    print(ds_.time)

# 電場データと磁場データの時間を合わせる

In [ ]:
def make_ds_EB_func(ds_E, E_vars, ds_B, B_vars, time_base, output_vars):
    ds_E_interp = ds_E.interp(time=time_base, method='linear')
    ds_B_interp = ds_B.interp(time=time_base, method='linear')

    da_Ex   = ds_E_interp[E_vars[0]]
    da_Ey   = ds_E_interp[E_vars[1]]
    da_Ez   = ds_E_interp[E_vars[2]]
    da_Bx   = ds_B_interp[B_vars[0]]
    da_By   = ds_B_interp[B_vars[1]]
    da_Bz   = ds_B_interp[B_vars[2]]

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

In [ ]:
Espin_vars  = ['Espin_gsm_x', 'Espin_gsm_y', 'Espin_gsm_z']
E8_vars     = ['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z']
E512_vars   = ['E512_gsm_x', 'E512_gsm_y', 'E512_gsm_z']
Bspin_vars  = ['Bspin_gsm_x', 'Bspin_gsm_y', 'Bspin_gsm_z']
B16_vars    = ['B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z']
B128_vars   = ['B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']
EBspin_vars = ['Espin_gsm_x', 'Espin_gsm_y', 'Espin_gsm_z', 'Bspin_gsm_x', 'Bspin_gsm_y', 'Bspin_gsm_z']
EB8_vars    = ['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B8_gsm_x', 'B8_gsm_y', 'B8_gsm_z']
EB128_vars  = ['E128_gsm_x', 'E128_gsm_y', 'E128_gsm_z', 'B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']

In [ ]:
ds_EBspin_gsm_segs  = []

ds_EBspin_gsm_segs.append(make_ds_EB_func(ds_Espin_data_gsm_segs[0], Espin_vars, ds_Bspin_data_gsm_segs[0], Bspin_vars, ds_Espin_data_gsm_segs[0].time, EBspin_vars).dropna(dim='time', how='all'))

print(ds_EBspin_gsm_segs)

In [ ]:
ds_EB8_gsm_segs = []

ds_EB8_gsm_segs.append(make_ds_EB_func(ds_E8_data_gsm_segs[0], E8_vars, ds_B16_data_gsm_segs[0], B16_vars, ds_E8_data_gsm_segs[0].time, EB8_vars).dropna(dim='time', how='all'))
ds_EB8_gsm_segs.append(make_ds_EB_func(ds_E8_data_gsm_segs[1], E8_vars, ds_B16_data_gsm_segs[2], B16_vars, ds_E8_data_gsm_segs[1].time, EB8_vars).dropna(dim='time', how='all'))
ds_EB8_gsm_segs.append(make_ds_EB_func(ds_E8_data_gsm_segs[2], E8_vars, ds_B16_data_gsm_segs[2], B16_vars, ds_E8_data_gsm_segs[2].time, EB8_vars).dropna(dim='time', how='all'))

print(ds_EB8_gsm_segs)

In [ ]:
ds_EB128_gsm_segs = []

ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[0], E512_vars, ds_B128_data_gsm_segs[0], B128_vars, ds_B128_data_gsm_segs[0].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[1], E512_vars, ds_B128_data_gsm_segs[1], B128_vars, ds_B128_data_gsm_segs[1].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[2], E512_vars, ds_B128_data_gsm_segs[2], B128_vars, ds_B128_data_gsm_segs[2].time, EB128_vars).dropna(dim='time', how='all'))

print(ds_EB128_gsm_segs)

In [ ]:
path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330'
os.makedirs(path_base_save_plot, exist_ok=True)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['Espin_gsm_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['Espin_gsm_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['Espin_gsm_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['Bspin_gsm_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['Bspin_gsm_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['Bspin_gsm_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_gsm_spin'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_gsm_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EBspin_gsm_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E8_gsm_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E8_gsm_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E8_gsm_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B8_gsm_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B8_gsm_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B8_gsm_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_gsm_8Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_gsm_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB8_gsm_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E128_gsm_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E128_gsm_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E128_gsm_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B128_gsm_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B128_gsm_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B128_gsm_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_gsm_128Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_gsm_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB128_gsm_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# FAC座標系を定義、DSI座標系 -> FAC座標系変換行列の作成

# FAC座標系の定義
- z軸は、背景磁場$B_{0}$の単位ベクトルで与える。
- x軸は、反地球方向かつz軸と垂直な単位ベクトルで与える。 ($E_{x}$: Toroidal component, $B_{x}$: Poloidal component)
- y軸は、z軸とx軸の外積で与える。 ($E_{y}$: Poloidal component, $B_{y}$: Toroidal component)

In [ ]:
psp.themis.state(probe='a', trange=time_range, no_update=True)

da_THA_pos_gsm  = pt.data_quants['tha_pos_gsm']
da_THA_pos_gsm  = da_THA_pos_gsm.sortby('time').sel(time=slice(time_range[0], time_range[1]))

da_THA_pos_unit_gsm = da_THA_pos_gsm / np.sqrt((da_THA_pos_gsm * da_THA_pos_gsm).sum(dim='v_dim'))

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
time_width_B_16Hz       = (B16_data_gsm.time[10] - B16_data_gsm.time[9]) / np.timedelta64(1, 's')
da_B_background         = B16_data_gsm.rolling(time=int(background_time_sec/time_width_B_16Hz), center=True).mean('time')
da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

print(da_B_background_unit)
print(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim')))
print(np.nanmin(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))
print(np.nanmax(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))

In [ ]:
time_array  = da_B_background_unit.time

da_THA_pos_unit_gsm_interp    = da_THA_pos_unit_gsm.interp(time=time_array)

da_u_   = da_THA_pos_unit_gsm_interp - (da_THA_pos_unit_gsm_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inGSM    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inGSM    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inGSM    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inGSM, da_e_x_FAC_inGSM, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

print(da_e_x_FAC_inGSM)
print('')
print(da_e_y_FAC_inGSM)
print('')
print(da_e_z_FAC_inGSM)

In [ ]:
R_FAC_to_GSM = xr.concat(
    [da_e_x_FAC_inGSM, da_e_y_FAC_inGSM, da_e_z_FAC_inGSM],
    dim='axis'
)
R_FAC_to_GSM    = R_FAC_to_GSM.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC']).assign_coords(v_dim=np.arange(3))
R_FAC_to_GSM    = R_FAC_to_GSM.dropna(dim='time', how='any')

R_GSM_to_FAC    = R_FAC_to_GSM.transpose('time', 'v_dim', 'axis')

print(R_FAC_to_GSM)
print('')
print(R_GSM_to_FAC)

In [ ]:
da_e_x_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=0)
da_e_y_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=1)
da_e_z_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=2)

In [ ]:
import os
import matplotlib.pyplot as plt

path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330/coordinate_FAC_GSM"
)
os.makedirs(path_base_save_plot, exist_ok=True)

def setup_ax(ax, xlabel, ylabel, title):
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(True, which='both', linestyle=':')
    ax.set_title(title)

def plot_fac_dsi_frame(it, frame_idx, save_dir):
    ex = da_e_x_GSM_inFAC.isel(time=it)
    ey = da_e_y_GSM_inFAC.isel(time=it)
    ez = da_e_z_GSM_inFAC.isel(time=it)

    # FAC 成分
    ex_x = ex.sel(axis='x_FAC').item()
    ex_y = ex.sel(axis='y_FAC').item()
    ex_z = ex.sel(axis='z_FAC').item()

    ey_x = ey.sel(axis='x_FAC').item()
    ey_y = ey.sel(axis='y_FAC').item()
    ey_z = ey.sel(axis='z_FAC').item()

    ez_x = ez.sel(axis='x_FAC').item()
    ez_y = ez.sel(axis='y_FAC').item()
    ez_z = ez.sel(axis='z_FAC').item()

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # ------------- (x, y) plane -------------
    ax = axs[0]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='k',   label='FAC-x')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='gray', label='FAC-y')
    ax.quiver(0, 0, 0, 0, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z', linewidth=0)

    ax.quiver(0, 0, ex_x, ex_y, angles='xy', scale_units='xy', scale=1,
              color='r', label='GSM-x')
    ax.quiver(0, 0, ey_x, ey_y, angles='xy', scale_units='xy', scale=1,
              color='b', label='GSM-y')
    ax.quiver(0, 0, ez_x, ez_y, angles='xy', scale_units='xy', scale=1,
              color='g', label='GSM-z')

    setup_ax(ax, 'FAC-x (Radial)', 'FAC-y (Longitudinal)', '(x, y) plane')
    ax.legend(loc='lower left')

    # ------------- (x, z) plane -------------
    ax = axs[1]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='k',   label='FAC-x')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z')

    ax.quiver(0, 0, ex_x, ex_z, angles='xy', scale_units='xy', scale=1,
              color='r', label='GSM-x')
    ax.quiver(0, 0, ey_x, ey_z, angles='xy', scale_units='xy', scale=1,
              color='b', label='GSM-y')
    ax.quiver(0, 0, ez_x, ez_z, angles='xy', scale_units='xy', scale=1,
              color='g', label='GSM-z')

    setup_ax(ax, 'FAC-x (Radial)', 'FAC-z (Parallel)', '(x, z) plane')

    # ------------- (y, z) plane -------------
    ax = axs[2]
    ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1,
              color='gray',   label='FAC-y')
    ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1,
              color='purple', label='FAC-z')

    ax.quiver(0, 0, ex_y, ex_z, angles='xy', scale_units='xy', scale=1,
              color='r', label='GSM-x')
    ax.quiver(0, 0, ey_y, ey_z, angles='xy', scale_units='xy', scale=1,
              color='b', label='GSM-y')
    ax.quiver(0, 0, ez_y, ez_z, angles='xy', scale_units='xy', scale=1,
              color='g', label='GSM-z')

    setup_ax(ax, 'FAC-y (Longitudinal)', 'FAC-z (Parallel)', '(y, z) plane')

    fig.suptitle(str(da_e_x_GSM_inFAC.time.values[it]), fontsize=14)
    plt.tight_layout()

    # ファイル名：time index をゼロ埋め
    fname = os.path.join(save_dir, f"coord_{frame_idx:06d}.png")
    fig.savefig(fname, dpi=150)
    plt.close(fig)


In [ ]:
#from concurrent.futures import ProcessPoolExecutor, as_completed
#import multiprocessing as mp
#from tqdm import tqdm
#import numpy as np
#
## --- 1) タスク作成 ---
#n_time = da_e_x_GSM_inFAC.sizes['time']
#step = 16 * 60   # 64Hz × 60 sec
#
#tasks = []
#frame_idx = 0
#for it in range(0, n_time, step):
#    tasks.append((it, frame_idx))
#    frame_idx += 1
#
#print("num frames:", len(tasks))
#
#
## --- 2) worker ---
#def worker(args):
#    it, frame_idx, save_path = args
#    plot_fac_dsi_frame(it, frame_idx, save_path)
#    return frame_idx
#
#
## --- 3) 並列 + tqdm ---
#save_path = path_base_save_plot
#n_workers = max(1, mp.cpu_count() - 1)
#
#with ProcessPoolExecutor(max_workers=n_workers) as exe:
#    futures = [
#        exe.submit(worker, (it, idx, save_path))
#        for it, idx in tasks
#    ]
#
#    for f in tqdm(as_completed(futures), total=len(futures)):
#        _ = f.result()   # 例外を拾うため

# GSM座標系 -> FAC座標系変換の実行

In [ ]:
ds_EBspin_fac_segs  = []

for ds_seg in ds_EBspin_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['Espin_gsm_x'], ds_seg['Espin_gsm_y'], ds_seg['Espin_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['Bspin_gsm_x'], ds_seg['Bspin_gsm_y'], ds_seg['Bspin_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'Espin_fac_x',
            'y_FAC': 'Espin_fac_y',
            'z_FAC': 'Espin_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'Bspin_fac_x',
            'y_FAC': 'Bspin_fac_y',
            'z_FAC': 'Bspin_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EBspin_fac_segs.append(ds_fac)

In [ ]:
ds_EB8_fac_segs = []

for ds_seg in ds_EB8_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E8_gsm_x'], ds_seg['E8_gsm_y'], ds_seg['E8_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B8_gsm_x'], ds_seg['B8_gsm_y'], ds_seg['B8_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E8_fac_x',
            'y_FAC': 'E8_fac_y',
            'z_FAC': 'E8_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B8_fac_x',
            'y_FAC': 'B8_fac_y',
            'z_FAC': 'B8_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB8_fac_segs.append(ds_fac)


In [ ]:
ds_EB128_fac_segs = []

for ds_seg in ds_EB128_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E128_gsm_x'], ds_seg['E128_gsm_y'], ds_seg['E128_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B128_gsm_x'], ds_seg['B128_gsm_y'], ds_seg['B128_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E128_fac_x',
            'y_FAC': 'E128_fac_y',
            'z_FAC': 'E128_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B128_fac_x',
            'y_FAC': 'B128_fac_y',
            'z_FAC': 'B128_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB128_fac_segs.append(ds_fac)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T22:25:00')
#t_end   = np.datetime64('2022-09-01T23:10:00')
#step_min    = 45
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E128_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E128_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E128_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B128_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B128_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B128_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_128Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB128_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['Espin_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['Espin_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['Espin_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['Bspin_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['Bspin_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['Bspin_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_spin'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EBspin_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E8_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E8_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E8_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B8_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B8_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B8_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_8Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB8_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

In [ ]:
#import os
#import numpy as np
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#path_base_save_plot = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
#)
#os.makedirs(path_base_save_plot, exist_ok=True)
#
#mpl.rcParams['font.size'] = 15
#
## ---- 5分刻みの時間窓 ----
#t_start = np.datetime64('2022-09-01T20:00:00')
#t_end   = np.datetime64('2022-09-02T00:00:00')
#step_min    = 1
#step    = np.timedelta64(step_min, 'm')
#
#t_list = []
#t0 = t_start
#while t0 < t_end:
#    t1 = t0 + step
#    t_list.append((t0, t1))
#    t0 = t1
#
#def plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir):
#    """全 seg を重ねて [t0, t1] の波形を描いて保存"""
#
#    fig = plt.figure(figsize=(10, 10))
#    gs = fig.add_gridspec(6, 1)
#    ax_0 = fig.add_subplot(gs[0, 0])
#    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    ax_1.tick_params(axis='x', which='both', labelbottom=False)
#    ax_2.tick_params(axis='x', which='both', labelbottom=False)
#    ax_3.tick_params(axis='x', which='both', labelbottom=False)
#    ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#    # ---- 各 seg を同じ軸に重ね描き ----
#    for ds_seg in ds_list:
#        ds_win = ds_seg.sel(time=slice(t0, t1))
#        if ds_win.time.size == 0:
#            continue
#
#        ax_0.plot(ds_win.time, ds_win['E128_fac_x'], lw=1, c='k')
#        ax_1.plot(ds_win.time, ds_win['E128_fac_y'], lw=1, c='k')
#        ax_2.plot(ds_win.time, ds_win['E128_fac_z'], lw=1, c='k')
#        ax_3.plot(ds_win.time, ds_win['B128_fac_x'], lw=1, c='k')
#        ax_4.plot(ds_win.time, ds_win['B128_fac_y'], lw=1, c='k')
#        ax_5.plot(ds_win.time, ds_win['B128_fac_z'], lw=1, c='k')
#
#    ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#    ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#    ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#    ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#    ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#    ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
#        ax.minorticks_on()
#        ax.grid(which='both', alpha=0.5)
#
#    ax_5.set_xlim(t0, t1)
#    fig.tight_layout()
#
#    if os.path.isdir(save_dir):
#        save_dir_   = f'{save_dir}/waveform/{step_min}min_fac_128Hz'
#        os.makedirs(save_dir_, exist_ok=True)
#        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
#        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
#        fname = f'EB_fields_dsi_{t0_str}_{t1_str}.png'
#        fpath = os.path.join(save_dir_, fname)
#        fig.savefig(fpath)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close()
#
#
#from concurrent.futures import ProcessPoolExecutor
#import multiprocessing as mp
#
#def worker(args):
#    """並列実行するためのラッパー"""
#    ds_list, t0, t1, save_dir = args
#    plot_EB_window_all_segs_fac(ds_list, t0, t1, save_dir)
#    return str(t0)  # ログ用
#
## 並列実行用のタスクをまとめる
#tasks = [(ds_EB128_fac_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]
#
## CPUコア数
#n_cores = max(1, mp.cpu_count())
#
#with ProcessPoolExecutor(max_workers=n_cores) as exe:
#    for out in exe.map(worker, tasks):
#        print("done:", out)

# 軌道データから、衛星速度(GSM)を導出

In [ ]:
v_sc_gsm    = pt.data_quants['tha_vel_gsm'].sortby('time')

In [ ]:
path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330"
)

In [ ]:
#import matplotlib.pyplot as plt
#import datetime
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_gsm_analysis  = v_sc_gsm.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_gsm_analysis.time, v_sc_gsm_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sc_gsm_analysis.time, v_sc_gsm_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sc_gsm_analysis.time, v_sc_gsm_analysis.data[:, 2], lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (GSM)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (GSM)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (GSM)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_sc_gsm_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_gsm.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
R_interp    = R_GSM_to_FAC.interp(time=v_sc_gsm.time)

v_sc_fac    = xr.dot(v_sc_gsm, R_interp, dims='v_dim')
v_sc_fac    = v_sc_fac.dropna(dim='time', how='any')
print(v_sc_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sc_fac_analysis  = v_sc_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sc_fac_analysis.time, v_sc_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(v_sc_fac_analysis.time, np.sqrt(v_sc_fac_analysis.data[:, 0]**2E0 + v_sc_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sc}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sc_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.themis.mom(trange=time_range, probe='a', level='l2', no_update=True)

In [ ]:
ND_electron     = pt.data_quants['tha_peem_density'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [/cc]
Temp_electron   = pt.data_quants['tha_peem_ptot'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [eV]

ND_ion          = pt.data_quants['tha_peim_density'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [/cc]
Temp_ion        = pt.data_quants['tha_peim_ptot'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all') # [eV]
v_ion_gsm       = pt.data_quants['tha_peim_velocity_gsm'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')    # [km/s]

In [ ]:
R_interp    = R_GSM_to_FAC.interp(time=v_ion_gsm.time)

v_ion_fac   = xr.dot(v_ion_gsm, R_interp, dims='v_dim')
v_ion_fac   = v_ion_fac.dropna(dim='time', how='any')

print(v_ion_fac)
print(v_sc_fac)

In [ ]:
v_sys_fac   = v_ion_fac - v_sc_fac.interp(time=v_ion_fac.time, method='linear')
v_sys_fac   = v_sys_fac.dropna(dim='time', how='any')
print(v_sys_fac)
print(v_sys_fac.time)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_gsm_analysis  = v_ion_gsm.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_gsm_analysis.time, v_ion_gsm_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_ion_gsm_analysis.time, v_ion_gsm_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_ion_gsm_analysis.time, v_ion_gsm_analysis.data[:, 2], lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (GSM)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (GSM)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (GSM)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(v_ion_gsm_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_gsm.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_ion_fac_analysis  = v_ion_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_ion_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/23:05:45', '20220901/23:08:15']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_v_ion_fac        = (v_ion_fac.time.data[1] - v_ion_fac.time.data[0]) / np.timedelta64(1, 's')
#v_ion_fac_mean      = v_ion_fac.rolling(time=int(100/dt_v_ion_fac), center=True).mean()
#v_ion_fac_analysis  = v_ion_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_ion_fac_analysis.time, v_ion_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(v_ion_fac_analysis.time, np.sqrt(v_ion_fac_analysis.data[:, 0]**2E0 + v_ion_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_ion_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac_mean_event3.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#v_sys_fac_analysis  = v_sys_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis.time, v_sys_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis.time, np.sqrt(v_sys_fac_analysis.data[:, 0]**2E0 + v_sys_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#import matplotlib.pyplot as plt
#
#import matplotlib as mpl
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 20
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_v_sys_fac        = (v_sys_fac.time.data[1] - v_sys_fac.time.data[0]) / np.timedelta64(1, 's')
#v_sys_fac_mean      = v_sys_fac.rolling(time=int(100/dt_v_sys_fac), center=True).mean()
#v_sys_fac_analysis_mean     = v_sys_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 0], lw=1, c='k')
#ax_1.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 1], lw=1, c='k')
#ax_2.plot(v_sys_fac_analysis_mean.time, v_sys_fac_analysis_mean.data[:, 2], lw=1, c='k')
#ax_3.plot(v_sys_fac_analysis_mean.time, np.sqrt(v_sys_fac_analysis_mean.data[:, 0]**2E0 + v_sys_fac_analysis_mean.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(v_sys_fac_analysis_mean.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac_mean.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

- Alfvén speed
```math
v_{\mathrm{A}} := \frac{B_{0}}{\sqrt{\mu_{0} n_{\mathrm{e}} m_{\mathrm{i}}}}
```
- Ion thermal speed
```math
v_{\mathrm{thi}} := \sqrt{\frac{2 T_{\mathrm{i}}}{m_{\mathrm{i}}}}
```
- Ion acoustic speed
```math
c_{\mathrm{s}} := \sqrt{\frac{T_{\mathrm{e}}}{m_{\mathrm{i}}}}
```
- Proton cyclotron frequency
```math
f_{\mathrm{p}} := \frac{1}{2 \pi} \frac{e B_{0}}{m_{\mathrm{p}}}
```
- Ion plasma beta
```math
\beta_{\mathrm{i}} := \frac{2 \mu_{0} n_{\mathrm{e}} T_{\mathrm{i}}}{B_{0}^{2}} = \left( \frac{v_{\mathrm{thi}}}{v_{\mathrm{A}}} \right)^{2}
```
- Ion-to-electron temperature ratio
```math
\tau := \frac{T_{\mathrm{i}}}{T_{\mathrm{e}}} = \frac{1}{2} \left( \frac{v_{\mathrm{thi}}}{c_{\mathrm{s}}} \right)^{2}
```

In [ ]:
B_total = pt.data_quants['tha_fgs_btotal'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')   # [nT]

In [ ]:
time_base   = B_total.time
print(time_base)

ND_electron_interp  = ND_electron.interp(time=time_base, method='linear')

Temp_electron_interp    = Temp_electron.interp(time=time_base, method='linear')
Temp_ion_interp         = Temp_ion.interp(time=time_base, method='linear')
v_ion_fac_interp        = v_ion_fac.interp(time=time_base, method='linear') * 1E3   # [m/s]
v_sys_fac_interp        = v_sys_fac.interp(time=time_base, method='linear') * 1E3   # [m/s]

v_ion_fac_perp_interp   = xr.DataArray(
    data=np.sqrt(v_ion_fac_interp.data[:, 0]**2E0 + v_ion_fac_interp.data[:, 1]**2E0),
    dims=['time'],
    coords={'time': v_ion_fac_interp.time},
    attrs=v_ion_fac_interp.attrs
)
v_sys_fac_perp_interp   = xr.DataArray(
    data=np.sqrt(v_sys_fac_interp.data[:, 0]**2E0 + v_sys_fac_interp.data[:, 1]**2E0),
    dims=['time'],
    coords={'time': v_sys_fac_interp.time},
    attrs=v_sys_fac_interp.attrs
)

proton_mass = 1.6726219e-27  # kg
elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

Alfven_speed        = B_total*1E-9 / np.sqrt(mu0 * ND_electron_interp*1E6 * proton_mass)
ion_thermal_speed   = np.sqrt(2E0 * Temp_ion_interp*elementary_charge / proton_mass)
ion_acoustic_speed  = np.sqrt(Temp_electron_interp*elementary_charge / proton_mass)

electron_mass_kg        = 9.1093837E-31
electron_thermal_speed  = np.sqrt(2E0 * Temp_electron_interp*elementary_charge / electron_mass_kg)

proton_cycl_freq    = elementary_charge * B_total*1E-9 / proton_mass / 2E0 / np.pi

ion_plasma_beta     = (ion_thermal_speed / Alfven_speed)**2E0
ion_to_electron_temp_ratio  = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0

# moving mean
dt_time_base            = (time_base.data[10] - time_base.data[9]) / np.timedelta64(1, 's')
print(dt_time_base)

Alfven_speed_mean       = Alfven_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_thermal_speed_mean  = ion_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
electron_thermal_speed_mean = electron_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_acoustic_speed_mean = ion_acoustic_speed.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_perp_mean     = v_sys_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_x_mean        = v_sys_fac_interp[:, 0].rolling(time=int(100/dt_time_base), center=True).mean()
v_sys_fac_y_mean        = v_sys_fac_interp[:, 1].rolling(time=int(100/dt_time_base), center=True).mean()

v_ion_fac_perp_mean     = v_ion_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()
v_ion_fac_x_mean        = v_ion_fac_interp[:, 0].rolling(time=int(100/dt_time_base), center=True).mean()
v_ion_fac_y_mean        = v_ion_fac_interp[:, 1].rolling(time=int(100/dt_time_base), center=True).mean()

ion_plasma_beta_mean            = ion_plasma_beta.rolling(time=int(100/dt_time_base), center=True).mean()
ion_to_electron_temp_ratio_mean = ion_to_electron_temp_ratio.rolling(time=int(100/dt_time_base), center=True).mean()
proton_cycl_freq_mean           = proton_cycl_freq.rolling(time=int(100/dt_time_base), center=True).mean()

# DataSet格納
ds_velocity_ms_perp = xr.Dataset(
    {
        'Alfven_speed':             Alfven_speed_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_ion_speed':           v_ion_fac_perp_mean,
        'perp_sys_speed':           v_sys_fac_perp_mean
    }
)
ds_velocity_ms_perp = ds_velocity_ms_perp

print(ds_velocity_ms_perp)

ds_velocity_ms_toroidal = xr.Dataset(
    {
        'Alfven_speed':             Alfven_speed_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_ion_speed':           v_ion_fac_x_mean,
        'perp_sys_speed':           v_sys_fac_x_mean
    }
)
ds_velocity_ms_toroidal = ds_velocity_ms_toroidal

print(ds_velocity_ms_toroidal)

ds_velocity_ms_poloidal = xr.Dataset(
    {
        'Alfven_speed':             Alfven_speed_mean,
        'ion_thermal_speed':        ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':       ion_acoustic_speed_mean,
        'perp_ion_speed':           v_ion_fac_y_mean,
        'perp_sys_speed':           v_sys_fac_y_mean
    }
)
ds_velocity_ms_poloidal = ds_velocity_ms_poloidal

print(ds_velocity_ms_poloidal)

ds_parameter = xr.Dataset(
    {
        'ion_plasma_beta':      ion_plasma_beta_mean,
        'i-e_temp_ratio':       ion_to_electron_temp_ratio_mean,
        'proton_cycl_freq_Hz':  proton_cycl_freq_mean,
        'number_density_cc':    ND_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_ion_eV':          Temp_ion_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_electron_eV':     Temp_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'B_total_nT':           B_total.rolling(time=int(100/dt_time_base), center=True).mean()
    }
)
ds_parameter = ds_parameter

print(ds_parameter)

In [ ]:
def dedup_and_sort(ds):
    ds = ds.sortby("time")
    t = ds["time"].values
    _, keep = np.unique(t, return_index=True)  # 先勝ちで一意化
    return ds.isel(time=np.sort(keep))

ds_parameter            = dedup_and_sort(ds_parameter)
ds_velocity_ms_perp     = dedup_and_sort(ds_velocity_ms_perp)
ds_velocity_ms_poloidal = dedup_and_sort(ds_velocity_ms_poloidal)
ds_velocity_ms_toroidal = dedup_and_sort(ds_velocity_ms_toroidal)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3,  lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_perp.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3,  lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_poloidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3,  lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary_toroidal.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 15))
#gs = fig.add_gridspec(7, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_cc'],   lw=1, c='k')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],         lw=1, c='k')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],    lw=1, c='k')
#ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['i-e_temp_ratio'],      lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta'],     lw=1, c='k')
#ax_5.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],          lw=1, c='k')
#ax_6.plot(ds_parameter_analysis.time, ds_parameter_analysis['proton_cycl_freq_Hz'], lw=1, c='k')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'     + '\n' + '[/cc]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$'     + '\n' + '[eV]')
#ax_2.set_ylabel(r'$T_{\mathrm{e}}$'     + '\n' + '[eV]')
#ax_3.set_ylabel(r'$\tau$')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$B_{0}$'              + '\n' + '[nT]')
#ax_6.set_ylabel(r'$f_{\mathrm{H}^{+}}$' + '\n' + '[Hz]')
#
#ax_3.set_yscale('log')
#ax_4.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#
#ax_6.set_xlim(ds_parameter_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'parameter_summary.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
psp.themis.state(trange=time_range, probe='a', no_update=True)
psp.cotrans(name_in='tha_pos_gsm', name_out='tha_pos_sm', coord_in='gsm', coord_out='sm')
THA_SM_pos = pt.data_quants['tha_pos_sm'].sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
print(THA_SM_pos)

THA_rmlatmlt_R = np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0 + THA_SM_pos.data[:, 2]**2E0) / 6378.1
THA_rmlatmlt_MLAT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 2], np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0)))
THA_rmlatmlt_MLT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 1], THA_SM_pos.data[:, 0])) / 15. + 12.

THA_rmlatmlt_L  = THA_rmlatmlt_R / np.cos(np.deg2rad(THA_rmlatmlt_MLAT))**2E0

print(np.nanmax(THA_rmlatmlt_R), np.nanmin(THA_rmlatmlt_R))
print(np.nanmax(THA_rmlatmlt_MLAT), np.nanmin(THA_rmlatmlt_MLAT))
print(np.nanmax(THA_rmlatmlt_MLT), np.nanmin(THA_rmlatmlt_MLT))

print(np.nanmax(THA_rmlatmlt_L), np.nanmin(THA_rmlatmlt_L))

```math
\theta_{\mathrm{GSM}} := \mathrm{arctan} \left( \frac{B_{\mathrm{GSM}z}}{\sqrt{B_{\mathrm{GSM}x}^{2} + B_{\mathrm{GSM}y}^{2}}} \right)
```
[Lui et al., 1999; Duan et al., 2011]

In [ ]:
theta_GSM = np.rad2deg(np.arctan(B16_data_gsm.data[:, 2] / np.sqrt(B16_data_gsm.data[:, 0]**2E0 + B16_data_gsm.data[:, 1]**2E0)))

da_theta_GSM = xr.DataArray(
    data=theta_GSM,
    dims=['time'],
    coords={'time': B16_data_gsm.time},
    name='theta_GSM_deg'
)

da_theta_dt  = (da_theta_GSM.time[1] - da_theta_GSM.time[0]) / np.timedelta64(1, 's')
da_theta_GSM = da_theta_GSM.rolling(time=int(100/da_theta_dt), center=True).mean()

da_theta_GSM

In [ ]:
da_B16_data_gsm_z = xr.DataArray(
    data=B16_data_gsm.data[:, 2],
    dims=['time'],
    coords={'time': B16_data_gsm.time},
    name='B16_gsm_z'
)

da_B16_data_gsm_z = da_B16_data_gsm_z.rolling(time=int(100/da_theta_dt), center=True).mean()

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_analysis = ds_velocity_ms_perp.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#da_theta_GSM_analysis   = da_theta_GSM.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#da_B16_data_gsm_z_analysis          = B16_data_gsm[:, 2].sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_poloidal_analysis    = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_toroidal_analysis    = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#import matplotlib.ticker as mticker
#from datetime import datetime
#import matplotlib.dates as mdates
#
#mpl.rcParams['font.size'] = 25
#
#fig = plt.figure(figsize=(11, 21))
#gs = fig.add_gridspec(7, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
##ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
##ax_6.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_cc'],           lw=1, c='k')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],                  lw=1, c='k', label=r'$B_{0}$')
##ax_2.plot(da_B16_data_gsm_z_analysis.time,  da_B16_data_gsm_z_analysis,                     lw=2, c='blue', label=r'$B_{\mathrm{z(GSM)}}$', alpha=0.4)
#ax_3.plot(da_theta_GSM_analysis.time, da_theta_GSM_analysis.data,                           lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta'],             lw=1, c='k')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=1, c='b')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k', label=r'$v_{\mathrm{A}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='red', label=r'$v_{\mathrm{thi}}$')
##ax_7.plot(ds_velocity_ms_toroidal_analysis.time, ds_velocity_ms_toroidal_analysis['perp_sys_speed']*1E-3, lw=1, c='orange', label=r'$v_{\mathrm{sys}x}$')
##ax_7.plot(ds_velocity_ms_poloidal_analysis.time, ds_velocity_ms_poloidal_analysis['perp_sys_speed']*1E-3, lw=1, c='green', label=r'$v_{\mathrm{sys}y}$')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + r'[$\mathrm{cm}^{-3}$]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
#ax_2.set_ylabel(r'$B_{0}$'                              + '\n' + '[nT]')
#ax_3.set_ylabel(r'$\theta_{\mathrm{GSM}}$'              + '\n' + '[deg]')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$v_{\mathrm{the}}$'                   + '\n' + '[km/s]')
#ax_6.set_ylabel(r'$v_{\mathrm{A}}$, $v_{\mathrm{thi}}$' + '\n' + '[km/s]')
##ax_7.set_ylabel(r'$v_{\mathrm{sys}\perp}$'              + '\n' + '[km/s]')
#
##ax_0.set_yscale('log')
##ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=0)
#ax_4.set_yscale('log')
##ax_5.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
##ax_7.minorticks_on()
##ax_7.grid(which='both', alpha=0.5)
#
#ax_1.legend(fontsize=20, ncol=2)
##ax_2.legend(fontsize=20, ncol=2)
#ax_6.legend(fontsize=20, ncol=2)
##ax_7.legend(fontsize=20, ncol=2)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_6.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#ax_6.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
#ax_6.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))
#
#def add_panel_label(ax, label, x=-0.15, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#def to_py_datetime(t_np64):
#    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)
#
## 軌道データ
#t_pos_py = to_py_datetime(THA_SM_pos.time.values)
#t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数
#
#R   = np.asarray(THA_rmlatmlt_R, dtype=float)  # Re
#mlat= np.asarray(THA_rmlatmlt_MLAT, dtype=float)  # deg
#mlt = np.asarray(THA_rmlatmlt_MLT, dtype=float)  # hour [0,24)
#
#L_shell = np.asarray(THA_rmlatmlt_L, dtype=float)
#
## --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
#mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)
#
## 補間関数（tick の x は「日数」なのでそのまま使う）
#def interp_at(x_num):
#    #Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
#    Ri    = np.interp(x_num, t_pos_num, L_shell, left=np.nan, right=np.nan)   # L-shell
#    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
#    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
#    mlti  = np.mod(mltiu, 24.0)
#    return Ri, mlati, mlti
#
## 目盛フォーマッタ
#def rmlt_formatter(x, pos=None):
#    Ri, mlati, mlti = interp_at(x)
#    if np.any(~np.isfinite([Ri, mlati, mlti])):
#        return ""  # 範囲外は空
#    return (f"{Ri:0.2f}\n"
#            f"{mlati:0.2f}\n"
#            f"{mlti:0.2f}")
#
## セカンダリ x 軸（底 side）を作ってラベルを差し替え
#secax = ax_6.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
#secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))
#
## メインの時間ラベルと重ならないよう余白を広げる
#ax_6.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
#secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル
#
## 好みで：目盛間隔をメイン x と合わせる
#secax.set_ticks(ax_6.get_xticks())
#
#fig.text(0.07, 0.067, "hhmm", ha='center', va='center')
##fig.text(0.07, 0.047, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
#fig.text(0.07, 0.047, r"L-shell", ha='center', va='center')
#fig.text(0.07, 0.027, r"MLAT", ha='center', va='center')
#fig.text(0.07, 0.007, r"MLT", ha='center', va='center')
#
#add_panel_label(ax_0, '(e)')
#add_panel_label(ax_1, '(f)')
#add_panel_label(ax_2, '(g)')
#add_panel_label(ax_3, '(h)')
#add_panel_label(ax_4, '(i)')
#add_panel_label(ax_5, '(j)')
#add_panel_label(ax_6, '(k)')
##add_panel_label(ax_7, '(l)')
#
#fig.suptitle('THEMIS-A', y=0.99)
#
#fig.subplots_adjust(hspace=0)
#fig.tight_layout(pad=0)
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'Figure_1_c.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    fig.savefig(os.path.join(path_base_save_plot, 'Figure_1_c.pdf'))
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# KAWの確認に適した時間窓$T_{\mathrm{window}}$の検討

```math
\frac{1}{v_{\mathrm{A}}} \frac{|\bf{E}_{\perp}|}{|\bf{B}_{\perp}|} = \frac{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2}}{\sqrt{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}} = \sqrt{10} \\
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
\therefore T_{\mathrm{window}} := \frac{1}{f_{\mathrm{sc}}} = \frac{1}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \left[ 9 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \left\{ 10 + \sqrt{117 \left( \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)^{2} + 180 \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} + 100} \right\} \right]^{-\frac{1}{2}}
```

In [ ]:
T_window = 1. / ds_parameter['proton_cycl_freq_Hz'].data * ds_velocity_ms_perp['ion_thermal_speed'].data / ds_velocity_ms_perp['perp_sys_speed'].data / np.sqrt(9. + 1. / ds_parameter['i-e_temp_ratio'].data * (10. + np.sqrt(117. / (ds_parameter['i-e_temp_ratio'].data)**(2.) + 180. / ds_parameter['i-e_temp_ratio'].data + 100.)))

da_T_window = xr.DataArray(data=T_window, dims=('time'), coords={'time': ds_velocity_ms_perp.time}, name='T_window')
da_T_window

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(da_T_window_analysis))

In [ ]:
#import matplotlib as mpl
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#mpl.rcParams['font.size'] = 15
#fig = plt.figure(figsize=(10, 4))
#ax = fig.add_subplot(111)
#ax.plot(da_T_window_analysis.time, da_T_window_analysis.data, c='k', lw=1)
#ax.minorticks_on()
#ax.set_ylabel(r'$T_{\mathrm{window}}$' + '\n[sec]')
#ax.set_yscale('log')
#ax.set_ylim(ymin=1)
#ax.grid(which='both', alpha=0.5)
#ax.set_xlim(da_T_window_analysis.time.values[[0, -1]])
#
#fig.tight_layout()
#
#print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'T_window.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# Wavelet analysis

In [ ]:
import os
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

import sys
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.tdwavelet_themis as tw
import importlib
importlib.reload(tw)

In [ ]:
import numpy as np
import xarray as xr
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import pandas as pd

def generate_red_noise(N, g=0.72):
    """AR(1)モデルによるレッドノイズの生成"""
    noise = np.random.randn(N)
    red_noise = np.zeros(N)
    for i in range(1, N):
        red_noise[i] = g * red_noise[i-1] + noise[i]
    return red_noise

BINS = np.linspace(0, 1, 1001)

def mc_worker_to_hist(idx, fs, n_points, g, dt, s0, dj, J):
    np.random.seed(idx)
    d1 = generate_red_noise(n_points, g=g)
    d2 = generate_red_noise(n_points, g=g)
    
    time = (np.arange(n_points) * dt * 1e9).astype('int64').astype('datetime64[ns]')
    ds_sim = xr.Dataset({"E": ("time", d1), "B": ("time", d2)}, coords={"time": time})
    
    ds_cwt = tw.cwt_from_dataset(ds_sim, dt=dt, s0=s0, dj=dj, J=J, variables=["E", "B"], apply_coi_mask=False)
    wco, _ = tw.calculate_xwt_wco(ds_cwt, "E_coef", "B_coef", dt, dj)
    
    mid = n_points // 4
    wco_target = wco[mid:-mid, :]
    
    # 各周波数ごとにヒストグラムを計算して返す
    # Shape: (n_freqs, 1000)
    hists = np.array([np.histogram(wco_target[:, f], bins=BINS)[0] for f in range(wco_target.shape[1])])
    return hists

def run_wco_monte_carlo(fs, n_iterations=1000, n_points=87168, g=0.72, n_jobs=-1, J=None):
    dt, s0, dj = 1.0/fs, 2.0, 1.0/32.0
    
    # Jが指定されていない場合のみ、点数から計算する
    if J is None:
        J = int(np.ceil(np.log2((n_points * dt / 3.0) / s0) / dj))
    
    # ダミーで周波数軸を取得
    time_coords = pd.to_datetime(np.arange(100) * dt, unit='s')
    dummy_ds = tw.cwt_from_dataset(
        xr.Dataset({"E": ("time", np.zeros(100)), "B": ("time", np.zeros(100))}, 
        coords={"time": time_coords}), 
        dt=dt, s0=s0, dj=dj, J=J, variables=["E","B"], apply_coi_mask=False
    )
    freqs = dummy_ds.freq.values
    n_freqs = len(freqs)
    
    # 全周波数の累積ヒストグラム (メモリ固定：数MB程度)
    # ※ BINS は外部で定義されている前提
    total_hists = np.zeros((n_freqs, len(BINS)-1), dtype=np.int64)

    print(f"Starting Robust Parallel MC: fs={fs}Hz, iterations={n_iterations}")
    
    # 小分けにして実行 (メモリの安全弁)
    batch_size = 40
    
    # tqdmで全体の進捗を管理
    with tqdm(total=n_iterations, desc="Monte Carlo Simulation") as pbar:
        for i in range(0, n_iterations, batch_size):
            current_batch = min(batch_size, n_iterations - i)
            
            # 並列実行
            results = Parallel(n_jobs=n_jobs)(
                delayed(mc_worker_to_hist)(i + j, fs, n_points, g, dt, s0, dj, J) 
                for j in range(current_batch)
            )
            
            # バッチ結果を累積
            for h in results:
                total_hists += h
            
            # プログレスバーを更新
            pbar.update(current_batch)
            
    # ヒストグラムから95%有意水準を逆算
    sig95_per_freq = []
    for f_idx in range(n_freqs):
        sum_hist = np.sum(total_hists[f_idx])
        if sum_hist == 0:
            sig95_per_freq.append(np.nan)
            continue
            
        cdf = np.cumsum(total_hists[f_idx]) / sum_hist
        # CDFが0.95を超える最初のビンの値を閾値とする
        idx95 = np.where(cdf >= 0.95)[0][0]
        sig95_per_freq.append(BINS[idx95])
        
    return freqs, np.array(sig95_per_freq)

#freqs, sig95 = run_wco_monte_carlo(fs=128.0, n_points=681*128, n_jobs=8, n_iterations=40)
#
#plt.figure(figsize=(14, 8))
#plt.semilogx(freqs, sig95, label='95% Significance Level')
#plt.axhline(np.mean(sig95), color='r', linestyle='--', label='Average')
#plt.xlabel('Frequency [Hz]')
#plt.ylabel('WCO Value')
#plt.title('Scale dependence of Significance Level (fs=128Hz)')
#plt.legend()
#plt.grid(True, which="both", ls="-", alpha=0.5)
#plt.show()

# 各データ(データ長)に合わせて、検証する必要がある。また、interation回数は1000以上にする。

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

# --- 設定 ---
SAVE_DIR = Path("/mnt/j/observation_data/THEMIS_analysis_save_data")
MC_CACHE_DIR = SAVE_DIR / "mc_cache"  # 点数ごとの有意水準を保存するフォルダ
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# パラメータ
fs = 0.3648
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 0.01         # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EBspin_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = f_min_phys #max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

EBspin_fac_vars = ['Espin_fac_x', 'Espin_fac_y', 'Espin_fac_z', 'Bspin_fac_x', 'Bspin_fac_y', 'Bspin_fac_z']
#
## --- セグメント処理ループ ---
#for i, ds_seg in enumerate(ds_EBspin_fac_segs):
#    n_points = len(ds_seg.time)
#    start_time_str = pd.to_datetime(ds_seg.time.values[0]).strftime('%Y%m%d_%H%M%S')
#    save_path = SAVE_DIR / f"themis_cwt_xwt_fs{fs:.4f}_seg{i}.nc"
#
#    # --- 1) このセグメントの点数に対応する有意水準を取得 ---
#    # ファイル名にパラメータを含めて衝突を防ぐ
#    mc_filename = MC_CACHE_DIR / f"sig95_fs{fs:.4f}_seg{i}_J{J_longest}.nc"
#    
#    if mc_filename.exists():
#        ds_sig = xr.open_dataset(mc_filename)
#    else:
#        print(f"[{i+1}] Calculating MC sig95 for n_points={n_points}...")
#        # J=J_longest を明示的に渡すことで、周波数軸の数を 388 に固定する
#        freqs_mc, sig95 = run_wco_monte_carlo(
#            fs=fs, n_iterations=1000, n_points=n_points, n_jobs=12, J=J_longest
#        )
#        ds_sig = xr.Dataset(
#            {"sig95": ("freq", sig95)},
#            coords={"freq": freqs_mc}
#        )
#        ds_sig.to_netcdf(mc_filename)
#    
#    sig95_values = ds_sig.sig95.values
#
#    # --- 2) CWT / WCO 計算 ---
#    ds_cwt = tw.cwt_from_dataset(
#        ds_seg, dt=dt, s0=s0, dj=dj, J=J_longest, variables=EBspin_fac_vars
#    )
#    
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'Espin_fac_x_coef', 'Bspin_fac_y_coef', dt, dj)
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'Espin_fac_y_coef', 'Bspin_fac_x_coef', dt, dj)
#
#    # --- 3) 結果の統合 ---
#    ds_cwt["EBspin_wco_exby"] = (("time", "freq"), wco_exby.astype(np.float32))
#    ds_cwt["EBspin_phase_exby"] = (("time", "freq"), phase_exby.astype(np.float32))
#    ds_cwt["EBspin_wco_eybx"] = (("time", "freq"), wco_eybx.astype(np.float32))
#    ds_cwt["EBspin_phase_eybx"] = (("time", "freq"), phase_eybx.astype(np.float32))
#    
#    # セグメントごとの有意水準も一緒に保存しておくと後で楽
#    # 周波数軸が一致していることを前提に、ブロードキャストして格納
#    ds_cwt["wco_sig95"] = (("freq"), sig95_values.astype(np.float32))
#
#    # 不要な複素数係数は除外して保存
#    vars_to_save = [v for v in ds_cwt.data_vars if not v.endswith('_coef')]
#    ds_cwt[vars_to_save].to_netcdf(save_path)
#    
#    print(f"[{i+1}/{len(ds_EBspin_fac_segs)}] Done: {save_path.name}")
#
#    # メモリ解放
#    del ds_cwt, ds_sig

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

# --- 設定 ---
SAVE_DIR = Path("/mnt/j/observation_data/THEMIS_analysis_save_data")
MC_CACHE_DIR = SAVE_DIR / "mc_cache"  # 点数ごとの有意水準を保存するフォルダ
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# パラメータ
fs = 8.0
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 0.01         # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB8_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = f_min_phys #max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

EB8_fac_vars = ['E8_fac_x', 'E8_fac_y', 'E8_fac_z', 'B8_fac_x', 'B8_fac_y', 'B8_fac_z']
#
## --- セグメント処理ループ ---
#for i, ds_seg in enumerate(ds_EB8_fac_segs):
#    n_points = len(ds_seg.time)
#    start_time_str = pd.to_datetime(ds_seg.time.values[0]).strftime('%Y%m%d_%H%M%S')
#    save_path = SAVE_DIR / f"themis_cwt_xwt_fs{fs:.4f}_seg{i}.nc"
#
#    # --- 1) このセグメントの点数に対応する有意水準を取得 ---
#    # ファイル名にパラメータを含めて衝突を防ぐ
#    mc_filename = MC_CACHE_DIR / f"sig95_fs{fs:.4f}_seg{i}_J{J_longest}.nc"
#    
#    if mc_filename.exists():
#        ds_sig = xr.open_dataset(mc_filename)
#    else:
#        print(f"[{i+1}] Calculating MC sig95 for n_points={n_points}...")
#        # J=J_longest を明示的に渡すことで、周波数軸の数を 388 に固定する
#        freqs_mc, sig95 = run_wco_monte_carlo(
#            fs=fs, n_iterations=1000, n_points=n_points, n_jobs=8, J=J_longest
#        )
#        ds_sig = xr.Dataset(
#            {"sig95": ("freq", sig95)},
#            coords={"freq": freqs_mc}
#        )
#        ds_sig.to_netcdf(mc_filename)
#    
#    sig95_values = ds_sig.sig95.values
#
#    # --- 2) CWT / WCO 計算 ---
#    ds_cwt = tw.cwt_from_dataset(
#        ds_seg, dt=dt, s0=s0, dj=dj, J=J_longest, variables=EB8_fac_vars
#    )
#    
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'E8_fac_x_coef', 'B8_fac_y_coef', dt, dj)
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'E8_fac_y_coef', 'B8_fac_x_coef', dt, dj)
#
#    # --- 3) 結果の統合 ---
#    ds_cwt["EB8_wco_exby"] = (("time", "freq"), wco_exby.astype(np.float32))
#    ds_cwt["EB8_phase_exby"] = (("time", "freq"), phase_exby.astype(np.float32))
#    ds_cwt["EB8_wco_eybx"] = (("time", "freq"), wco_eybx.astype(np.float32))
#    ds_cwt["EB8_phase_eybx"] = (("time", "freq"), phase_eybx.astype(np.float32))
#    
#    # セグメントごとの有意水準も一緒に保存しておくと後で楽
#    # 周波数軸が一致していることを前提に、ブロードキャストして格納
#    ds_cwt["wco_sig95"] = (("freq"), sig95_values.astype(np.float32))
#
#    # 不要な複素数係数は除外して保存
#    vars_to_save = [v for v in ds_cwt.data_vars if not v.endswith('_coef')]
#    ds_cwt[vars_to_save].to_netcdf(save_path)
#    
#    print(f"[{i+1}/{len(ds_EB8_fac_segs)}] Done: {save_path.name}")
#
#    # メモリ解放
#    del ds_cwt, ds_sig

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

# --- 設定 ---
SAVE_DIR = Path("/mnt/j/observation_data/THEMIS_analysis_save_data")
MC_CACHE_DIR = SAVE_DIR / "mc_cache"  # 点数ごとの有意水準を保存するフォルダ
SAVE_DIR.mkdir(parents=True, exist_ok=True)
MC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# パラメータ
fs = 128.0
dt = 1.0 / fs
s0 = 2.0
dj = 1.0/32.0

f_target = 4.0          # 目標最低周波数
N_cycle_min = 3.0       # 最低3周期は欲しい

# ---- 1) 全セグメントの長さを調べる ----
T_list = []
for ds_seg in ds_EB128_fac_segs:
    t0 = ds_seg.time.values[0]
    t1 = ds_seg.time.values[-1]
    T_seg = (t1 - t0) / np.timedelta64(1, 's')
    T_list.append(T_seg)

T_max = max(T_list)
print("max segment length [s] =", T_max)

# ---- 2) 最長セグメントで意味のある最低周波数 ----
f_min_phys = N_cycle_min / T_max
f_min_seg = f_min_phys #max(f_target, f_min_phys)
print("f_min_seg (for longest segment) =", f_min_seg)

# ---- 3) その周波数に対応する scale_max ----
scale_target = 1.0 / (f_min_seg * dt)

# ---- 4) J_longest を計算 ----
J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
print("J_longest =", J_longest)

EB128_fac_vars = ['E128_fac_x', 'E128_fac_y', 'E128_fac_z', 'B128_fac_x', 'B128_fac_y', 'B128_fac_z']
#
## --- セグメント処理ループ ---
#for i, ds_seg in enumerate(ds_EB128_fac_segs):
#    if i == 0:
#        continue
#    n_points = len(ds_seg.time)
#    start_time_str = pd.to_datetime(ds_seg.time.values[0]).strftime('%Y%m%d_%H%M%S')
#    save_path = SAVE_DIR / f"themis_cwt_xwt_fs{fs:.4f}_seg{i}.nc"
#
#    # --- 1) このセグメントの点数に対応する有意水準を取得 ---
#    # ファイル名にパラメータを含めて衝突を防ぐ
#    mc_filename = MC_CACHE_DIR / f"sig95_fs{fs:.4f}_seg{i}_J{J_longest}.nc"
#    
#    if mc_filename.exists():
#        ds_sig = xr.open_dataset(mc_filename)
#    else:
#        print(f"[{i+1}] Calculating MC sig95 for n_points={n_points}...")
#        # J=J_longest を明示的に渡すことで、周波数軸の数を 388 に固定する
#        freqs_mc, sig95 = run_wco_monte_carlo(
#            fs=fs, n_iterations=1000, n_points=n_points, n_jobs=8, J=J_longest
#        )
#        ds_sig = xr.Dataset(
#            {"sig95": ("freq", sig95)},
#            coords={"freq": freqs_mc}
#        )
#        ds_sig.to_netcdf(mc_filename)
#    
#    sig95_values = ds_sig.sig95.values
#
#    # --- 2) CWT / WCO 計算 ---
#    ds_cwt = tw.cwt_from_dataset(
#        ds_seg, dt=dt, s0=s0, dj=dj, J=J_longest, variables=EB128_fac_vars
#    )
#    
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'E128_fac_x_coef', 'B128_fac_y_coef', dt, dj)
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'E128_fac_y_coef', 'B128_fac_x_coef', dt, dj)
#
#    # --- 3) 結果の統合 ---
#    ds_cwt["EB128_wco_exby"] = (("time", "freq"), wco_exby.astype(np.float32))
#    ds_cwt["EB128_phase_exby"] = (("time", "freq"), phase_exby.astype(np.float32))
#    ds_cwt["EB128_wco_eybx"] = (("time", "freq"), wco_eybx.astype(np.float32))
#    ds_cwt["EB128_phase_eybx"] = (("time", "freq"), phase_eybx.astype(np.float32))
#    
#    # セグメントごとの有意水準も一緒に保存しておくと後で楽
#    # 周波数軸が一致していることを前提に、ブロードキャストして格納
#    ds_cwt["wco_sig95"] = (("freq"), sig95_values.astype(np.float32))
#
#    # 不要な複素数係数は除外して保存
#    vars_to_save = [v for v in ds_cwt.data_vars if not v.endswith('_coef')]
#    ds_cwt[vars_to_save].to_netcdf(save_path)
#    
#    print(f"[{i+1}/{len(ds_EB128_fac_segs)}] Done: {save_path.name}")
#
#    # メモリ解放
#    del ds_cwt, ds_sig

In [ ]:
#import xarray as xr
#import matplotlib.pyplot as plt
#from pathlib import Path
#from tqdm.auto import tqdm
#
## --- 設定 ---
#target_dir = Path("/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/")
#
## ディレクトリ内のすべての .nc ファイルを取得
#nc_files = list(target_dir.glob("*.nc"))
#
#print(f"Found {len(nc_files)} files in {target_dir}")
#
#mpl.rcParams['font.size']=15
#
## 各ファイルに対して処理を実行
#for nc_path in tqdm(nc_files, desc="Generating plots"):
#    # 出力ファイルパスの生成 (.nc -> .png)
#    png_path = nc_path.with_suffix(".png")
#
#    try:
#        # データの読み込み
#        with xr.open_dataset(nc_path) as ds:
#            freq = ds.freq.values
#            sig95 = ds.sig95.values
#            
#            # 図の作成
#            plt.figure(figsize=(10, 5))
#            plt.semilogx(freq, sig95, color='red', lw=2, label='95% Significance Level')
#
#            plt.grid(True, which="both", ls="-", alpha=0.5)
#            plt.xlabel('Frequency [Hz]')
#            plt.ylabel('WCO Significance Threshold')
#            plt.title(f'95% Significance Level vs. Frequency\n{nc_path.name}')
#            plt.legend()
#
#            # 保存
#            plt.savefig(png_path, dpi=150, bbox_inches='tight')
#            plt.close() # メモリ解放のために閉じる
#
#    except Exception as e:
#        print(f"Error processing {nc_path.name}: {e}")
#
#print("Done. All plots saved in the same folder.")
#
#mpl.rcParams['font.size']=25

In [ ]:
#import xarray as xr
#
## パスの定義
#path_seg = "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs128.0000_seg0.nc"
#path_mc = "/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs128.0000_seg0_J443.nc"
#
## ファイルを開く
#try:
#    ds_seg = xr.open_dataset(path_seg)
#    ds_mc = xr.open_dataset(path_mc)
#
#    print("--- Segment Data (CWT/XWT) ---")
#    print(ds_seg)
#    print(ds_seg.time)
#    print(f"\n'freq' size in segment: {ds_seg.dims.get('freq')}")
#    
#    print("\n" + "="*50 + "\n")
#    
#    print("--- Monte Carlo Cache (sig95) ---")
#    print(ds_mc)
#    print(f"\n'freq' size in MC cache: {ds_mc.dims.get('freq')}")
#
#    # サイズの不一致を直接確認
#    if ds_seg.dims.get('freq') != ds_mc.dims.get('freq'):
#        print("\n[ALERT] Dimension mismatch detected!")
#        print(f"Segment freq ({ds_seg.dims.get('freq')}) != MC freq ({ds_mc.dims.get('freq')})")
#
#except FileNotFoundError as e:
#    print(f"Error: ファイルが見つからない。パスを確認してくれ。\n{e}")
#except Exception as e:
#    print(f"Error: ロード中に問題が発生した。\n{e}")

In [ ]:
#import matplotlib.pyplot as plt
#
#freq = ds_mc.freq.values
#sig95 = ds_mc.sig95.values
#
#plt.figure(figsize=(10, 5))
#plt.semilogx(freq, sig95, color='red', lw=2, label='95% Significance Level')
#
#plt.grid(True, which="both", ls="-", alpha=0.5)
#plt.xlabel('Frequency [Hz]')
#plt.ylabel('WCO Significance Threshold')
##plt.ylim((0, 0.05))
#plt.title('95% Significance Level vs. Frequency')
#plt.legend()
#
## 値の範囲を表示して確認
#print(f"Max Sig95: {sig95.max():.4f} at {freq[sig95.argmax()]:.2e} Hz")
#print(f"Min Sig95: {sig95.min():.4f} at {freq[sig95.argmin()]:.2e} Hz")
#
#plt.show()

In [ ]:
#fs = 0.3648
#dt = 1.0 / fs
#s0 = 2.0
#dj = 1.0/32.0
#
#f_target = 0.01         # 目標最低周波数
#N_cycle_min = 3.0       # 最低3周期は欲しい
#
## ---- 1) 全セグメントの長さを調べる ----
#T_list = []
#for ds_seg in ds_EB8_fac_segs:
#    t0 = ds_seg.time.values[0]
#    t1 = ds_seg.time.values[-1]
#    T_seg = (t1 - t0) / np.timedelta64(1, 's')
#    T_list.append(T_seg)
#
#T_max = max(T_list)
#print("max segment length [s] =", T_max)
#
## ---- 2) 最長セグメントで意味のある最低周波数 ----
#f_min_phys = N_cycle_min / T_max
#f_min_seg = f_min_phys #max(f_target, f_min_phys)
#print("f_min_seg (for longest segment) =", f_min_seg)
#
## ---- 3) その周波数に対応する scale_max ----
#scale_target = 1.0 / (f_min_seg * dt)
#
## ---- 4) J_longest を計算 ----
#J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
#print("J_longest =", J_longest)
#
#ds_EBspin_fac_cwt_segs    = []
#EBspin_fac_vars = ['Espin_fac_x', 'Espin_fac_y', 'Espin_fac_z', 'Bspin_fac_x', 'Bspin_fac_y', 'Bspin_fac_z']
#
#for ds_seg in ds_EBspin_fac_segs:
#    ds_cwt = tw.cwt_from_dataset(
#        ds_seg,
#        dt=dt,
#        s0=s0,
#        dj=dj,
#        J=J_longest,     # << 全てこれを使う
#        variables=EBspin_fac_vars
#    )
#    ds_EBspin_fac_cwt_segs.append(ds_cwt)
#
#print(ds_EBspin_fac_cwt_segs)
#
#ds_EBspin_fac_xwt_wco_segs = []
#
#def process_segment_EBspin(ds_cwt, dt, dj):
#
#    # XWT / WCO計算
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'Espin_fac_x_coef', 'Bspin_fac_y_coef', dt, dj)
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'Espin_fac_y_coef', 'Bspin_fac_x_coef', dt, dj)
#
#    # 結果の格納
#    ds_xwt = xr.Dataset({
#        "EBspin_wco_exby": (("time", "freq"), wco_exby.astype(np.float32)),
#        "EBspin_phase_exby": (("time", "freq"), phase_exby.astype(np.float32)),
#        "EBspin_wco_eybx": (("time", "freq"), wco_eybx.astype(np.float32)),
#        "EBspin_phase_eybx": (("time", "freq"), phase_eybx.astype(np.float32))
#    }, coords=ds_cwt.coords)
#
#    return ds_xwt
#
#for ds_seg in ds_EBspin_fac_cwt_segs:
#    ds_xwt = process_segment_EBspin(ds_seg, dt, dj)
#    ds_EBspin_fac_xwt_wco_segs.append(ds_xwt)
#
#print(ds_EBspin_fac_xwt_wco_segs)

In [ ]:
#fs = 8.0
#dt = 1.0 / fs
#s0 = 2.0
#dj = 1.0/32.0
#
#f_target = 0.01         # 目標最低周波数
#N_cycle_min = 3.0       # 最低3周期は欲しい
#
## ---- 1) 全セグメントの長さを調べる ----
#T_list = []
#for ds_seg in ds_EB8_fac_segs:
#    t0 = ds_seg.time.values[0]
#    t1 = ds_seg.time.values[-1]
#    T_seg = (t1 - t0) / np.timedelta64(1, 's')
#    T_list.append(T_seg)
#
#T_max = max(T_list)
#print("max segment length [s] =", T_max)
#
## ---- 2) 最長セグメントで意味のある最低周波数 ----
#f_min_phys = N_cycle_min / T_max
#f_min_seg = f_min_phys #max(f_target, f_min_phys)
#print("f_min_seg (for longest segment) =", f_min_seg)
#
## ---- 3) その周波数に対応する scale_max ----
#scale_target = 1.0 / (f_min_seg * dt)
#
## ---- 4) J_longest を計算 ----
#J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
#print("J_longest =", J_longest)
#
#ds_EB8_fac_cwt_segs    = []
#EB8_fac_vars = ['E8_fac_x', 'E8_fac_y', 'E8_fac_z', 'B8_fac_x', 'B8_fac_y', 'B8_fac_z']
#
#ds_EB8_fac_cwt_segs = []
#for ds_seg in ds_EB8_fac_segs:
#    ds_cwt = tw.cwt_from_dataset(
#        ds_seg, dt=dt, s0=s0, dj=dj, J=J_longest, variables=EB8_fac_vars
#    )
#    ds_EB8_fac_cwt_segs.append(ds_cwt)
#
#print(ds_EB8_fac_cwt_segs)
#
#ds_EB8_fac_xwt_wco_segs = []
#
#def process_segment_EB8(ds_cwt, dt, dj):
#
#    # XWT / WCO計算
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'E8_fac_x_coef', 'B8_fac_y_coef', dt, dj)
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'E8_fac_y_coef', 'B8_fac_x_coef', dt, dj)
#
#    # 結果の格納
#    ds_xwt = xr.Dataset({
#        "EB8_wco_exby": (("time", "freq"), wco_exby.astype(np.float32)),
#        "EB8_phase_exby": (("time", "freq"), phase_exby.astype(np.float32)),
#        "EB8_wco_eybx": (("time", "freq"), wco_eybx.astype(np.float32)),
#        "EB8_phase_eybx": (("time", "freq"), phase_eybx.astype(np.float32))
#    }, coords=ds_cwt.coords)
#
#    return ds_xwt
#
#for ds_seg in ds_EB8_fac_cwt_segs:
#    ds_xwt = process_segment_EB8(ds_seg, dt, dj)
#    ds_EB8_fac_xwt_wco_segs.append(ds_xwt)
#
#print(ds_EB8_fac_xwt_wco_segs)

In [ ]:
#fs = 128.0
#dt = 1.0 / fs
#s0 = 2.0
#dj = 1.0/32.0
#
#f_target = 4.0          # 目標最低周波数
#N_cycle_min = 3.0       # 最低3周期は欲しい
#
## ---- 1) 全セグメントの長さを調べる ----
#T_list = []
#for ds_seg in ds_EB128_fac_segs:
#    t0 = ds_seg.time.values[0]
#    t1 = ds_seg.time.values[-1]
#    T_seg = (t1 - t0) / np.timedelta64(1, 's')
#    T_list.append(T_seg)
#
#T_max = max(T_list)
#print("max segment length [s] =", T_max)
#
## ---- 2) 最長セグメントで意味のある最低周波数 ----
#f_min_phys = N_cycle_min / T_max
#f_min_seg = f_min_phys #max(f_target, f_min_phys)
#print("f_min_seg (for longest segment) =", f_min_seg)
#
## ---- 3) その周波数に対応する scale_max ----
#scale_target = 1.0 / (f_min_seg * dt)
#
## ---- 4) J_longest を計算 ----
#J_longest = int(np.ceil(np.log2(scale_target / s0) / dj))
#print("J_longest =", J_longest)
#
#ds_EB128_fac_cwt_segs    = []
#EB128_fac_vars = ['E128_fac_x', 'E128_fac_y', 'E128_fac_z', 'B128_fac_x', 'B128_fac_y', 'B128_fac_z']
#
#for ds_seg in ds_EB128_fac_segs:
#    ds_cwt = tw.cwt_from_dataset(
#        ds_seg,
#        dt=dt,
#        s0=s0,
#        dj=dj,
#        J=J_longest,     # << 全てこれを使う
#        variables=EB128_fac_vars
#    )
#    ds_EB128_fac_cwt_segs.append(ds_cwt)
#
#print(ds_EB128_fac_cwt_segs)
#
#ds_EB128_fac_xwt_wco_segs  = []
#
#for ds_seg in ds_EB128_fac_cwt_segs:
#    # 1) Ex - By ペア
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_seg, 'E128_fac_x_coef', 'B128_fac_y_coef', dt, dj)
#    
#    # 2) Ey - Bx ペア
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_seg, 'E128_fac_y_coef', 'B128_fac_x_coef', dt, dj)
#    
#    # 結果を Dataset に格納
#    ds_xwt = xr.Dataset({
#        "EB128_wco_exby": (("time", "freq"), wco_exby.astype(np.float32)),
#        "EB128_phase_exby": (("time", "freq"), phase_exby.astype(np.float32)),
#        "EB128_wco_eybx": (("time", "freq"), wco_eybx.astype(np.float32)),
#        "EB128_phase_eybx": (("time", "freq"), phase_eybx.astype(np.float32))
#    })
#
#    ds_EB128_fac_xwt_wco_segs.append(ds_xwt)
#
#print(ds_EB128_fac_xwt_wco_segs)
#
#ds_EB128_fac_xwt_wco_segs = []
#
#def process_segment_EB128(ds_cwt, dt, dj):
#
#    # XWT / WCO計算
#    wco_exby, phase_exby = tw.calculate_xwt_wco(ds_cwt, 'E128_fac_x_coef', 'B128_fac_y_coef', dt, dj)
#    wco_eybx, phase_eybx = tw.calculate_xwt_wco(ds_cwt, 'E128_fac_y_coef', 'B128_fac_x_coef', dt, dj)
#
#    # 結果の格納
#    ds_xwt = xr.Dataset({
#        "EB128_wco_exby": (("time", "freq"), wco_exby.astype(np.float32)),
#        "EB128_phase_exby": (("time", "freq"), phase_exby.astype(np.float32)),
#        "EB128_wco_eybx": (("time", "freq"), wco_eybx.astype(np.float32)),
#        "EB128_phase_eybx": (("time", "freq"), phase_eybx.astype(np.float32))
#    }, coords=ds_cwt.coords)
#
#    return ds_xwt
#
#for ds_seg in ds_EB128_fac_cwt_segs:
#    ds_xwt = process_segment_EB128(ds_seg, dt, dj)
#    ds_EB128_fac_xwt_wco_segs.append(ds_xwt)
#
#print(ds_EB128_fac_xwt_wco_segs)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
import matplotlib as mpl

mpl.rcParams['font.size'] = 11

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    # dsets をリストに正規化
    if isinstance(dsets, xr.Dataset):
        dsets = [dsets]
    elif isinstance(dsets, (str, bytes)):
        raise TypeError("dsets は Dataset のリストにして")

    das = []
    for ds in dsets:
        if ds is None or not isinstance(ds, xr.Dataset):
            continue
        if var in ds.data_vars:
            das.append(ds[var])

    if not das:
        return None, None

    pow_cat = xr.concat(das, dim="time").sortby("time")

    coi_name = var.replace("_cwt", "_coi")
    coi_list = []
    for ds in dsets:
        if isinstance(ds, xr.Dataset) and coi_name in ds.data_vars:
            coi_list.append(ds[coi_name])
    coi_cat = xr.concat(coi_list, dim="time").sortby("time") if coi_list else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
        #ax.set_xlim(t0, t1)
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    ax.grid(which='both', alpha=0.3)

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [ ]:
targets_EB128_fac_cwt = [
    ("E128_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B128_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB128_fac_cwt = {}

path_seg_EB128_list = [
    "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs128.0000_seg0.nc",
    "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs128.0000_seg1.nc",
    "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs128.0000_seg2.nc"
]

ds_EB128_fac_cwt_segs = []

for path in path_seg_EB128_list:
    ds_seg = xr.open_dataset(path)
    ds_EB128_fac_cwt_segs.append(ds_seg)

for v, _, _ in targets_EB128_fac_cwt:
    da, coi = concat_cwt_segments(ds_EB128_fac_cwt_segs, v)
    if da is not None: joined_EB128_fac_cwt[v] = (da, coi)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(9)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB128_fac_cwt), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EB128_fac_cwt):
#        if v not in joined_EB128_fac_cwt: continue
#        da, coi = joined_EB128_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(np.nanmin(da.freq), np.nanmax(da.freq)),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_high_freq_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
targets_EB8_fac_cwt = [
    ("E8_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E8_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E8_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B8_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B8_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B8_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB8_fac_cwt = {}

path_seg_EB8_list = [
    "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs8.0000_seg0.nc",
    "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs8.0000_seg1.nc",
    "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs8.0000_seg2.nc"
]

ds_EB8_fac_cwt_segs = []

for path in path_seg_EB8_list:
    ds_seg = xr.open_dataset(path)
    ds_EB8_fac_cwt_segs.append(ds_seg)

for v, _, _ in targets_EB8_fac_cwt:
    da, coi = concat_cwt_segments(ds_EB8_fac_cwt_segs, v)
    if da is not None: joined_EB8_fac_cwt[v] = (da, coi)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T20:00') + np.timedelta64(5, 'm')*n
#    for n in range(48)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB8_fac_cwt), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EB8_fac_cwt):
#        if v not in joined_EB8_fac_cwt: continue
#        da, coi = joined_EB8_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(np.nanmin(da.freq), np.nanmax(da.freq)),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_low_freq_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
targets_EBspin_fac_cwt = [
    ("Espin_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("Espin_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("Espin_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("Bspin_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("Bspin_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("Bspin_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EBspin_fac_cwt = {}

path_seg_EBspin_list = [
    "/mnt/j/observation_data/THEMIS_analysis_save_data/themis_cwt_xwt_fs0.3648_seg0.nc"
]

ds_EBspin_fac_cwt_segs = []

for path in path_seg_EBspin_list:
    ds_seg = xr.open_dataset(path)
    ds_EBspin_fac_cwt_segs.append(ds_seg)

for v, _, _ in targets_EBspin_fac_cwt:
    da, coi = concat_cwt_segments(ds_EBspin_fac_cwt_segs, v)
    if da is not None: joined_EBspin_fac_cwt[v] = (da, coi)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T20:00') + np.timedelta64(5, 'm')*n
#    for n in range(48)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EBspin_fac_cwt), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets_EBspin_fac_cwt):
#        if v not in joined_EBspin_fac_cwt: continue
#        da, coi = joined_EBspin_fac_cwt[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(np.nanmin(da.freq), np.nanmax(da.freq)),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_spin_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

mpl.rcParams['font.size'] = 11

# --- 連結（周波数を合わせて縦結合）
def concat_cwt_segments(dsets, var):
    das = [ds[var] for ds in dsets if var in ds]
    if not das: return None, None
    freqs = np.unique(np.concatenate([da.freq.values for da in das]))
    das = [da if np.array_equal(da.freq.values, freqs) else da.interp(freq=freqs) for da in das]
    pow_cat = xr.concat(das, dim="time").sortby("time").assign_coords(freq=("freq", freqs))
    coi_name = var.replace("_cwt", "_coi")
    coi_cat = xr.concat([ds[coi_name] for ds in dsets if coi_name in ds], dim="time").sortby("time") \
              if all(coi_name in ds for ds in dsets) else None
    return pow_cat, coi_cat

# --- 2帯域を同一axに描画
def plot_cwt_dualband_on_ax(ax,
                            da_hi, coi_hi,   # 高周波 (128Hz系)
                            da_lo, coi_lo,   # 低周波 (8Hz系)
                            t0, t1, minutes,
                            f_lo=(1e-2, 4.0),
                            f_hi=(4.0, 64.0),
                            zrange=(1e-6, 1e3),
                            cmap="turbo",
                            ylabel="", unit_right="",
                            cax=None):

    if t1 is None:
        t1 = t0 + np.timedelta64(minutes, "m")

    # ---- データ切り出し ----
    dah  = da_hi.sel(time=slice(t0, t1)) if da_hi  is not None else None
    dal  = da_lo.sel(time=slice(t0, t1)) if da_lo  is not None else None
    coih = coi_hi.sel(time=slice(t0, t1)) if coi_hi is not None else None
    coil = coi_lo.sel(time=slice(t0, t1)) if coi_lo is not None else None

    if (dah is None or dah.time.size == 0) and (dal is None or dal.time.size == 0):
        return None, None

    def _prep(da, coi, fmin, fmax):
        if da is None or da.time.size == 0:
            return None, None, None
        da2 = da.sel(freq=slice(fmin, fmax))
        T = mdates.date2num(da2.time.values)
        F = da2.freq.values
        Z = da2.values.astype(float)
        if coi is not None:
            C = coi.values[:, None]
            Z = np.where(F[None, :] < C, np.nan, Z)
        Tm = np.tile(T, (F.size, 1)).T
        Fm = np.tile(F, (T.size, 1))
        return Tm, Fm, Z

    pcm_h = pcm_l = None

    # 低周波 (8 Hz 系)
    out_lo = _prep(dal, coil, *f_lo)
    if out_lo[0] is not None:
        Tm_l, Fm_l, Z_l = out_lo
        pcm_l = ax.pcolormesh(Tm_l, Fm_l, Z_l, shading="auto",
                              norm=LogNorm(vmin=zrange[0], vmax=zrange[1]),
                              cmap=cmap)

    # 高周波 (128 Hz 系)
    out_hi = _prep(dah, coih, *f_hi)
    if out_hi[0] is not None:
        Tm_h, Fm_h, Z_h = out_hi
        pcm_h = ax.pcolormesh(Tm_h, Fm_h, Z_h, shading="auto",
                              norm=LogNorm(vmin=zrange[0], vmax=zrange[1]),
                              cmap=cmap)

    # ---- 軸設定 ----
    ax.set_yscale("log")
    ax.set_ylim(f_lo[0], f_hi[1])
    ax.set_ylabel(f"{ylabel}\n[Hz]")

    ax.axhline(y=f_lo[1], c='k', alpha=0.5, lw=2)

    # t0〜t1 をそのまま xlim に
    ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.minorticks_on()
    ax.grid(which='both', alpha=0.3)

    # ---- カラーバー ----
    if cax == None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    pcm_main = pcm_h if pcm_h is not None else pcm_l
    cb = plt.colorbar(pcm_main, cax=cax)
    cb.set_label(unit_right)

    return (pcm_h, pcm_l), cb

In [ ]:
#targets_EB128_EB8 = [
#    ("E128_fac_x_cwt","E8_fac_x_cwt",  r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_y_cwt","E8_fac_y_cwt",  r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_z_cwt","E8_fac_z_cwt",  r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B128_fac_x_cwt","B8_fac_x_cwt",  r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_y_cwt","B8_fac_y_cwt",  r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_z_cwt","B8_fac_z_cwt",  r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#
#joined_EB128_EB8_fac_cwt = {}
#for v_hi, v_lo, _, _ in targets_EB128_EB8:
#    da_hi, coi_hi = concat_cwt_segments(ds_EB128_fac_cwt_segs, v_hi)
#    da_lo, coi_lo = concat_cwt_segments(ds_EB8_fac_cwt_segs,   v_lo)
#    if (da_hi is not None) or (da_lo is not None):
#        joined_EB128_EB8_fac_cwt[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)

In [ ]:
targets_EB128_EBspin = [
    ("E128_fac_x_cwt","Espin_fac_x_cwt",  r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_y_cwt","Espin_fac_y_cwt",  r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_z_cwt","Espin_fac_z_cwt",  r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B128_fac_x_cwt","Bspin_fac_x_cwt",  r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_y_cwt","Bspin_fac_y_cwt",  r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_z_cwt","Bspin_fac_z_cwt",  r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
]

joined_EB128_EBspin_fac_cwt = {}
for v_hi, v_lo, _, _ in targets_EB128_EBspin:
    da_hi, coi_hi = concat_cwt_segments(ds_EB128_fac_cwt_segs,  v_hi)
    da_lo, coi_lo = concat_cwt_segments(ds_EBspin_fac_cwt_segs, v_lo)
    if (da_hi is not None) or (da_lo is not None):
        joined_EB128_EBspin_fac_cwt[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(12)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB128_EBspin), 1, figsize=(10, 10), sharex=True)
#    for ax, (v_hi, v_lo, ylab, unit) in zip(axes, targets_EB128_EBspin):
#        if (v_hi, v_lo) not in joined_EB128_EBspin_fac_cwt: continue
#        da_hi, coi_hi, da_lo, coi_lo = joined_EB128_EBspin_fac_cwt[(v_hi, v_lo)]
#        plot_cwt_dualband_on_ax(ax, da_hi, coi_hi, da_lo, coi_lo,
#                                t0=t0, t1=None, minutes=5,
#                                f_lo=(1e-2, 0.1824), f_hi=(0.1824, 64.0),
#                                zrange=(1e-6, 1e3), cmap="turbo",
#                                ylabel=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

mpl.rcParams['font.size'] = 11

def plot_cwt_prefer_hi_fill_lo(ax,
                              da_hi, coi_hi,
                              da_lo, coi_lo,
                              t0, t1, minutes,
                              f_range=(1e-2, 64.0),
                              zrange=(1e-6, 1e3),
                              cmap="turbo",
                              ylabel="", unit_right="", cax=None):

    if t1 is None:
        t1 = t0 + np.timedelta64(minutes, "m")

    # 切り出し（NoneならNoneのまま）
    dah  = da_hi.sel(time=slice(t0, t1)) if da_hi  is not None else None
    dal  = da_lo.sel(time=slice(t0, t1)) if da_lo  is not None else None
    coih = coi_hi.sel(time=slice(t0, t1)) if coi_hi is not None else None

    # 両方ないなら終了
    if (dah is None or dah.time.size == 0) and (dal is None or dal.time.size == 0):
        return None, None

    # --- 共通 time グリッド（和集合） ---
    times = []
    if dah is not None and dah.time.size > 0:
        times.append(dah.time.values)
    if dal is not None and dal.time.size > 0:
        times.append(dal.time.values)
    t_grid = np.unique(np.concatenate(times))
    t_grid = np.sort(t_grid)

    # --- 共通 freq グリッド（128優先、なければspin） ---
    if dah is not None and dah.time.size > 0:
        f_grid = dah.sel(freq=slice(*f_range)).freq.values
    else:
        f_grid = dal.sel(freq=slice(*f_range)).freq.values

    # --- 128側を共通グリッドへ（無ければNaNで埋める） ---
    if dah is not None and dah.time.size > 0:
        dah2 = dah.sel(freq=slice(*f_range)).interp(time=t_grid, freq=f_grid)
    else:
        dah2 = xr.DataArray(
            np.full((t_grid.size, f_grid.size), np.nan, dtype=float),
            dims=("time","freq"),
            coords={"time": t_grid, "freq": f_grid},
            name="hi"
        )

    # --- spin側を共通グリッドへ（無ければNaN） ---
    if dal is not None and dal.time.size > 0:
        dal2 = dal.sel(freq=slice(*f_range)).interp(time=t_grid, freq=f_grid)
    else:
        dal2 = xr.full_like(dah2, np.nan)

    # --- COI(128) を共通 time へ（128が無い時刻は NaN にして境界線を切る） ---
    if coih is not None and coih.time.size > 0:
        # まず time 補間（coih は (time)）
        coih1 = coih.interp(time=t_grid)
        # 128が元々存在しない時刻を判定して、その時刻のcoiをNaNに
        # (nearestで突っ張られるのを防ぐ)
        hi_has_time = xr.DataArray(np.isin(t_grid, dah.time.values if dah is not None else []),
                                   dims=("time",), coords={"time": t_grid})
        coih1 = coih1.where(hi_has_time, np.nan)

        coih2 = coih1.astype(float).broadcast_like(dah2)
        hi_valid = (dah2["freq"] >= coih2) & np.isfinite(dah2)
    else:
        hi_valid = np.isfinite(dah2)

    # --- 合成：128優先、無効部はspin ---
    Z = dah2.where(hi_valid, dal2)

    # --- 描画 ---
    T = mdates.date2num(Z.time.values)
    F = Z.freq.values
    pcm = ax.pcolormesh(T, F, Z.T.values, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]),
                        cmap=cmap)

    # 境界線（128がある時刻だけ描く）
    if coih is not None and coih.time.size > 0:
        coih_plot = coih.interp(time=t_grid)
        coih_plot = coih_plot.where(np.isin(t_grid, dah.time.values), np.nan)
        ax.plot(mdates.date2num(coih_plot.time.values), coih_plot.values,
                "k", lw=2, alpha=0.7)

    ax.set_yscale("log")
    ax.set_ylim(f_range[0], f_range[1])
    ax.set_ylabel(f"{ylabel}\n[Hz]")
    ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.minorticks_on()
    ax.grid(which="both", alpha=0.3)

    if cax == None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(unit_right)

    return pcm, cb


In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(12)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB128_EBspin), 1, figsize=(10, 10), sharex=True)
#    for ax, (v_hi, v_lo, ylab, unit) in zip(axes, targets_EB128_EBspin):
#        if (v_hi, v_lo) not in joined_EB128_EBspin_fac_cwt: continue
#        da_hi, coi_hi, da_lo, coi_lo = joined_EB128_EBspin_fac_cwt[(v_hi, v_lo)]
#        plot_cwt_prefer_hi_fill_lo(
#            ax,
#            da_hi, coi_hi,
#            da_lo, coi_lo,
#            t0=t0, t1=None, minutes=5,
#            f_range=(1e-2, 64.0),
#            zrange=(1e-6, 1e3),
#            cmap="turbo",
#            ylabel=ylab, unit_right=unit
#        )
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_prefer128_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
mpl.rcParams['font.size'] = 11

def concat_xwt_segments(dsets, var):
    das = []
    for ds in dsets:
        if ds is not None and var in ds.data_vars:
            das.append(ds[var])
    if not das: return None
    return xr.concat(das, dim="time").sortby("time")

from matplotlib.colors import Normalize

def plot_xwt_phase_on_ax(ax, da_wco, da_phase, t0=None, minutes=5,
                         wco_thresh=0, yrange=(1e-2, 4.0),
                         mode="wco", label_left="", unit=None, cax=None):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        wco = da_wco.sel(time=slice(t0, t1))
        phase = da_phase.sel(time=slice(t0, t1))
    else:
        wco, phase = da_wco, da_phase

    if wco.time.size == 0: return None

    T = mdates.date2num(wco.time.values)
    F = wco.freq.values
    Tm, Fm = np.meshgrid(T, F, indexing='ij')

    if mode == "wco":
        Z = wco.values.astype(float)
        cmap = "viridis"
        norm = Normalize(vmin=0, vmax=1)
        if unit==None:
            unit = "Coherency"
    else: # mode == "phase"
        # WCOが低い領域をマスクする
        Z = np.abs(phase.where(wco > wco_thresh).values.astype(float))
        cmap = "Spectral"
        norm = Normalize(vmin=0, vmax=180)
        if unit==None:
            unit = "Phase (abs) [deg]"

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto", norm=norm, cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    ax.grid(which='both', alpha=0.3)

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)

    if cax == None:
        cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(unit)
    if mode == "phase":
        cb.set_ticks([0, 45, 90, 135, 180])
    
    return pcm, cb

In [ ]:
targets_EBspin_xwt = [
    ("EBspin_wco_exby", "EBspin_phase_exby", r"$E_{x}$"+r' & '+r"$B_{y}$"),
    ("EBspin_wco_eybx", "EBspin_phase_eybx", r"$E_{y}$"+r' & '+r"$B_{x}$"),
]

joined_EBspin_xwt = {}

path_seg_EBspin_mc_list = [
    "/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs0.3648_seg0_J313.nc"
]

for i, path in enumerate(path_seg_EBspin_mc_list):
    ds_seg_mc = xr.open_dataset(path)

    sig95 = ds_seg_mc["sig95"]

    ds = ds_EBspin_fac_cwt_segs[i]

    sig95_on_ds = sig95.copy()
    sig95_on_ds = sig95_on_ds.assign_coords(freq=ds["freq"])  # freq座標を強制的に一致させる

    sig95_2d = sig95_on_ds.broadcast_like(ds["EBspin_wco_exby"])

    m_exby = ds["EBspin_wco_exby"] >= sig95_2d
    m_eybx = ds["EBspin_wco_eybx"] >= sig95_2d

    ds = ds.assign(
        #EBspin_wco_exby   = ds["EBspin_wco_exby"].where(m_exby),
        EBspin_phase_exby = ds["EBspin_phase_exby"].where(m_exby),
        #EBspin_wco_eybx   = ds["EBspin_wco_eybx"].where(m_eybx),
        EBspin_phase_eybx = ds["EBspin_phase_eybx"].where(m_eybx),
    )

    ds_EBspin_fac_cwt_segs[i] = ds

for w_var, p_var, lab in targets_EBspin_xwt:
    w_da = concat_xwt_segments(ds_EBspin_fac_cwt_segs, w_var)
    p_da = concat_xwt_segments(ds_EBspin_fac_cwt_segs, p_var)
    if w_da is not None:
        joined_EBspin_xwt[lab] = (w_da, p_da)

print(joined_EBspin_xwt)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T20:00') + np.timedelta64(5, 'm')*n
#    for n in range(48)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
#    
#    for i, (w_var, p_var, lab) in enumerate(targets_EBspin_xwt):
#        if lab not in joined_EBspin_xwt: continue
#        w_da, p_da = joined_EBspin_xwt[lab]
#        
#        # WCOプロット
#        plot_xwt_phase_on_ax(axes[2*i], w_da, p_da, t0=t0, mode="wco",
#                             label_left=f"Coherency ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#        
#        # Phaseプロット
#        plot_xwt_phase_on_ax(axes[2*i+1], w_da, p_da, t0=t0, mode="phase",
#                             label_left=f"Phase ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_spin_xwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
targets_EB8_xwt = [
    ("EB8_wco_exby", "EB8_phase_exby", r"$E_{x}$"+r' & '+r"$B_{y}$"),
    ("EB8_wco_eybx", "EB8_phase_eybx", r"$E_{y}$"+r' & '+r"$B_{x}$"),
]

joined_EB8_xwt = {}

path_seg_EB8_mc_list = [
    "/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs8.0000_seg0_J437.nc",
    '/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs8.0000_seg1_J437.nc',
    '/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs8.0000_seg2_J437.nc'
]

for i, path in enumerate(path_seg_EB8_mc_list):
    ds_seg_mc = xr.open_dataset(path)

    sig95 = ds_seg_mc["sig95"]

    ds = ds_EB8_fac_cwt_segs[i]

    sig95_on_ds = sig95.copy()
    sig95_on_ds = sig95_on_ds.assign_coords(freq=ds["freq"])  # freq座標を強制的に一致させる

    sig95_2d = sig95_on_ds.broadcast_like(ds["EB8_wco_exby"])

    m_exby = ds["EB8_wco_exby"] >= sig95_2d
    m_eybx = ds["EB8_wco_eybx"] >= sig95_2d

    ds = ds.assign(
        #EB8_wco_exby   = ds["EB8_wco_exby"].where(m_exby),
        EB8_phase_exby = ds["EB8_phase_exby"].where(m_exby),
        #EB8_wco_eybx   = ds["EB8_wco_eybx"].where(m_eybx),
        EB8_phase_eybx = ds["EB8_phase_eybx"].where(m_eybx),
    )

    ds_EB8_fac_cwt_segs[i] = ds

for w_var, p_var, lab in targets_EB8_xwt:
    w_da = concat_xwt_segments(ds_EB8_fac_cwt_segs, w_var)
    p_da = concat_xwt_segments(ds_EB8_fac_cwt_segs, p_var)
    if w_da is not None:
        joined_EB8_xwt[lab] = (w_da, p_da)

print(joined_EB8_xwt)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T20:00') + np.timedelta64(5, 'm')*n
#    for n in range(48)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
#    
#    for i, (w_var, p_var, lab) in enumerate(targets_EB8_xwt):
#        if lab not in joined_EB8_xwt: continue
#        w_da, p_da = joined_EB8_xwt[lab]
#        
#        # WCOプロット
#        plot_xwt_phase_on_ax(axes[2*i], w_da, p_da, t0=t0, mode="wco",
#                             label_left=f"Coherency ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#        
#        # Phaseプロット
#        plot_xwt_phase_on_ax(axes[2*i+1], w_da, p_da, t0=t0, mode="phase",
#                             label_left=f"Phase ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_low_xwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
targets_EB128_xwt = [
    ("EB128_wco_exby", "EB128_phase_exby", r"$E_{x}$"+r' & '+r"$B_{y}$"),
    ("EB128_wco_eybx", "EB128_phase_eybx", r"$E_{y}$"+r' & '+r"$B_{x}$"),
]

joined_EB128_xwt = {}

path_seg_EB128_mc_list = [
    "/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs128.0000_seg0_J443.nc",
    '/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs128.0000_seg1_J443.nc',
    '/mnt/j/observation_data/THEMIS_analysis_save_data/mc_cache/sig95_fs128.0000_seg2_J443.nc'
]

for i, path in enumerate(path_seg_EB128_mc_list):
    ds_seg_mc = xr.open_dataset(path)

    sig95 = ds_seg_mc["sig95"]

    ds = ds_EB128_fac_cwt_segs[i]

    sig95_on_ds = sig95.copy()
    sig95_on_ds = sig95_on_ds.assign_coords(freq=ds["freq"])  # freq座標を強制的に一致させる

    sig95_2d = sig95_on_ds.broadcast_like(ds["EB128_wco_exby"])

    m_exby = ds["EB128_wco_exby"] >= sig95_2d
    m_eybx = ds["EB128_wco_eybx"] >= sig95_2d

    ds = ds.assign(
        #EB128_wco_exby   = ds["EB128_wco_exby"].where(m_exby),
        EB128_phase_exby = ds["EB128_phase_exby"].where(m_exby),
        #EB128_wco_eybx   = ds["EB128_wco_eybx"].where(m_eybx),
        EB128_phase_eybx = ds["EB128_phase_eybx"].where(m_eybx),
    )

    ds_EB128_fac_cwt_segs[i] = ds

for w_var, p_var, lab in targets_EB128_xwt:
    w_da = concat_xwt_segments(ds_EB128_fac_cwt_segs, w_var)
    p_da = concat_xwt_segments(ds_EB128_fac_cwt_segs, p_var)
    if w_da is not None:
        joined_EB128_xwt[lab] = (w_da, p_da)

print(joined_EB128_xwt)

In [ ]:
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(9)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
#    
#    for i, (w_var, p_var, lab) in enumerate(targets_EB128_xwt):
#        if lab not in joined_EB128_xwt: continue
#        w_da, p_da = joined_EB128_xwt[lab]
#        
#        # WCOプロット
#        plot_xwt_phase_on_ax(axes[2*i], w_da, p_da, t0=t0, mode="wco",
#                             label_left=f"Coherency ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#        
#        # Phaseプロット
#        plot_xwt_phase_on_ax(axes[2*i+1], w_da, p_da, t0=t0, mode="phase",
#                             label_left=f"Phase ({lab})", yrange=(np.nanmin(p_da.freq), np.nanmax(p_da.freq)))
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_high_xwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# 5分毎のPSDのMedianをplot

In [ ]:
da_Espin_fac_x_cwt  = joined_EBspin_fac_cwt["Espin_fac_x_cwt"]
da_Espin_fac_y_cwt  = joined_EBspin_fac_cwt["Espin_fac_y_cwt"]
da_Espin_fac_z_cwt  = joined_EBspin_fac_cwt["Espin_fac_z_cwt"]
da_Bspin_fac_x_cwt  = joined_EBspin_fac_cwt["Bspin_fac_x_cwt"]
da_Bspin_fac_y_cwt  = joined_EBspin_fac_cwt["Bspin_fac_y_cwt"]
da_Bspin_fac_z_cwt  = joined_EBspin_fac_cwt["Bspin_fac_z_cwt"]

da_E8_fac_x_cwt     = joined_EB8_fac_cwt["E8_fac_x_cwt"]
da_E8_fac_y_cwt     = joined_EB8_fac_cwt["E8_fac_y_cwt"]
da_E8_fac_z_cwt     = joined_EB8_fac_cwt["E8_fac_z_cwt"]
da_B8_fac_x_cwt     = joined_EB8_fac_cwt["B8_fac_x_cwt"]
da_B8_fac_y_cwt     = joined_EB8_fac_cwt["B8_fac_y_cwt"]
da_B8_fac_z_cwt     = joined_EB8_fac_cwt["B8_fac_z_cwt"]

da_E128_fac_x_cwt   = joined_EB128_fac_cwt["E128_fac_x_cwt"]
da_E128_fac_y_cwt   = joined_EB128_fac_cwt["E128_fac_y_cwt"]
da_E128_fac_z_cwt   = joined_EB128_fac_cwt["E128_fac_z_cwt"]
da_B128_fac_x_cwt   = joined_EB128_fac_cwt["B128_fac_x_cwt"]
da_B128_fac_y_cwt   = joined_EB128_fac_cwt["B128_fac_y_cwt"]
da_B128_fac_z_cwt   = joined_EB128_fac_cwt["B128_fac_z_cwt"]

In [ ]:
#import numpy as np
#import xarray as xr
#import matplotlib.pyplot as plt
#import os
#import matplotlib as mpl
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 20
#
## ---- 入力 ----
#pairs_spin = [
#    ("E_spin_FAC_x_cwt",    da_Espin_fac_x_cwt),
#    ("E_spin_FAC_y_cwt",    da_Espin_fac_y_cwt),
#    ("E_spin_FAC_z_cwt",    da_Espin_fac_z_cwt),
#    ("B_spin_FAC_x_cwt",    da_Bspin_fac_x_cwt),
#    ("B_spin_FAC_y_cwt",    da_Bspin_fac_y_cwt),
#    ("B_spin_FAC_z_cwt",    da_Bspin_fac_z_cwt),
#]
#
#pairs_8 = [
#    ("E_8_FAC_x_cwt",   da_E8_fac_x_cwt),
#    ("E_8_FAC_y_cwt",   da_E8_fac_y_cwt),
#    ("E_8_FAC_z_cwt",   da_E8_fac_z_cwt),
#    ("B_8_FAC_x_cwt",   da_B8_fac_x_cwt),
#    ("B_8_FAC_y_cwt",   da_B8_fac_y_cwt),
#    ("B_8_FAC_z_cwt",   da_B8_fac_z_cwt),
#]
#
#pairs_128 = [
#    ("E_128_FAC_x_cwt", da_E128_fac_x_cwt),
#    ("E_128_FAC_y_cwt", da_E128_fac_y_cwt),
#    ("E_128_FAC_z_cwt", da_E128_fac_z_cwt),
#    ("B_128_FAC_x_cwt", da_B128_fac_x_cwt),
#    ("B_128_FAC_y_cwt", da_B128_fac_y_cwt),
#    ("B_128_FAC_z_cwt", da_B128_fac_z_cwt),
#]
#
#t_all_start = np.datetime64('2022-09-01T20:00:00')
#t_all_end   = np.datetime64('2022-09-02T00:00:00')
#step        = np.timedelta64(5, 'm')  # 5分
#outdir      = (
#    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330/wavelet_PSD/5min_PSD"
#)
#os.makedirs(outdir, exist_ok=True)
#
## 事前に CWT 本体のみ取り出し、timeでソート
#pairs_spin_sorted = []
#for name, da in pairs_spin:
#    if isinstance(da, tuple):
#        da = da[0]
#    if isinstance(da, xr.DataArray):
#        pairs_spin_sorted.append((name, da.sortby('time')))
#
#pairs_8_sorted = []
#for name, da in pairs_8:
#    if isinstance(da, tuple):
#        da = da[0]
#    if isinstance(da, xr.DataArray):
#        pairs_8_sorted.append((name, da.sortby('time')))
#
#pairs_128_sorted = []
#for name, da in pairs_128:
#    if isinstance(da, tuple):
#        da = da[0]
#    if isinstance(da, xr.DataArray):
#        pairs_128_sorted.append((name, da.sortby('time')))
#
#def compute_median_dict(pairs_sorted, t0, t1):
#    d = {}
#    for name, da in pairs_sorted:
#        sub = da.sel(time=slice(t0, t1))
#        if sub.sizes.get('time', 0) == 0:
#            continue
#        if np.iscomplexobj(sub.data):
#            sub = (sub.real**2 + sub.imag**2)
#        d[name] = sub.median(dim='time', skipna=True)  # (freq,)
#    return d
#
#def plot_median_dict(mdict_1, mdict_2, mdict_3, t0, t1, outdir=None):
#    fig, ax = plt.subplots(figsize=(8, 8))
#
#    color_map = {
#        ('E', 'x'): 'blue',      # Ex
#        ('E', 'y'): 'orange',    # Ey
#        ('E', 'z'): 'brown',     # Ez
#        ('B', 'x'): 'green',     # Bx
#        ('B', 'y'): 'red',       # By
#        ('B', 'z'): 'purple',    # Bz
#    }
#
#    for name, med in mdict_1.items():
#        if name.split('_')[3] == 'z':
#            continue
#        prefix  = name.split('_')[0]
#        freq    = name.split('_')[1]
#        comp    = name.split('_')[3]
#        coor    = name.split('_')[2]
#        label  = f"${prefix}_{comp}$ ({coor})"
#        #label   = f"${prefix}_{comp} $"# ({coor}, {freq})"
#        col     = color_map.get((prefix, comp), 'gray')
#        ax.loglog(med['freq'], med, label=label, lw=1, color=col)
#
#    #for name, med in mdict_2.items():
#    #    prefix  = name.split('_')[0]
#    #    freq    = name.split('_')[1]
#    #    comp    = name.split('_')[3]
#    #    coor    = name.split('_')[2]
#    #    label  = f"${prefix}_{comp}$ ({coor}, {freq} Hz)"
#    #    ax.loglog(med['freq'], med, label=label)
#
#    for name, med in mdict_3.items():
#        if name.split('_')[3] == 'z':
#            continue
#        prefix  = name.split('_')[0]
#        freq    = name.split('_')[1]
#        comp    = name.split('_')[3]
#        coor    = name.split('_')[2]
#        label  = f"${prefix}_{comp}$ ({coor}, {freq} Hz)"
#        col     = color_map.get((prefix, comp), 'gray')
#        ax.loglog(med['freq'][med['freq']>0.3648/2.], med[med['freq']>0.3648/2.], lw=1, color=col)
#    
#    ax.axvline(0.3648/2., c='k', lw=1)
#
#    ax.minorticks_on()
#    ax.set_xlabel('Frequency [Hz]')
#    ax.set_ylabel('Median PSD')
#    ax.set_title(f"Median {str(t0)[11:]}–{str(t1)[11:]}\n (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)")
#    ax.grid(True, which='both', ls=':')
#    ax.set_yticks([1E-8, 1E-7, 1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
#    ax.legend(ncol=2, fontsize=15)
#    ax.set_xlim(1e-2, 64)
#    ax.set_ylim(1e-8, 1e4)
#    plt.tight_layout()
#
#    if outdir and os.path.isdir(outdir):
#        fn = f"median_bs_E_{str(t0).replace(':','')}_{str(t1).replace(':','')}.png"
#        fig.savefig(os.path.join(outdir, fn), dpi=300, bbox_inches='tight')
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)
###
#### ---- 5分窓でループ ----
###t_starts = np.arange(t_all_start, t_all_end, step)  # 21:00, 21:05, ..., 23:25
###for t0 in t_starts:
###    t1 = t0 + step
###    mdict_spin  = compute_median_dict(pairs_spin_sorted, t0, t1)
###    mdict_8     = compute_median_dict(pairs_8_sorted, t0, t1)
###    mdict_128   = compute_median_dict(pairs_128_sorted, t0, t1)
###    if not mdict_spin and not mdict_8 and not mdict_128:  # その窓でデータ無し
###        continue
###    plot_median_dict(mdict_spin, mdict_8, mdict_128, t0, t1, outdir=outdir)
##
#t_starts    = np.datetime64('2022-09-01T23:05:45')
#t_ends      = np.datetime64('2022-09-01T23:08:15')
#mdict_spin  = compute_median_dict(pairs_spin_sorted, t_starts, t_ends)
#mdict_128   = compute_median_dict(pairs_128_sorted, t_starts, t_ends)
#plot_median_dict(mdict_spin, None, mdict_128, t_starts, t_ends, outdir)

In [ ]:
mu_0 = 4.*np.pi*1E-7

Ex_fac  = xr.concat([ds_EB128_fac_segs[0]['E128_fac_x'], ds_EB128_fac_segs[1]['E128_fac_x'], ds_EB128_fac_segs[2]['E128_fac_x']], dim='time')
Ey_fac  = xr.concat([ds_EB128_fac_segs[0]['E128_fac_y'], ds_EB128_fac_segs[1]['E128_fac_y'], ds_EB128_fac_segs[2]['E128_fac_y']], dim='time')
Ez_fac  = xr.concat([ds_EB128_fac_segs[0]['E128_fac_z'], ds_EB128_fac_segs[1]['E128_fac_z'], ds_EB128_fac_segs[2]['E128_fac_z']], dim='time')
Bx_fac  = xr.concat([ds_EB128_fac_segs[0]['B128_fac_x'], ds_EB128_fac_segs[1]['B128_fac_x'], ds_EB128_fac_segs[2]['B128_fac_x']], dim='time')
By_fac  = xr.concat([ds_EB128_fac_segs[0]['B128_fac_y'], ds_EB128_fac_segs[1]['B128_fac_y'], ds_EB128_fac_segs[2]['B128_fac_y']], dim='time')
Bz_fac  = xr.concat([ds_EB128_fac_segs[0]['B128_fac_z'], ds_EB128_fac_segs[1]['B128_fac_z'], ds_EB128_fac_segs[2]['B128_fac_z']], dim='time')

S_para  = (Ex_fac * By_fac - Ey_fac * Bx_fac) / mu_0 * 1E-12

S_para_toroidal = Ex_fac * By_fac / mu_0 * 1E-12
S_para_poloidal = - Ey_fac * Bx_fac / mu_0 * 1E-12

print(S_para)
print(S_para_toroidal)
print(S_para_poloidal)

In [ ]:
mu_0 = 4.*np.pi*1E-7

Ex_fac_spin  = xr.concat([ds_EBspin_fac_segs[0]['Espin_fac_x']], dim='time')
Ey_fac_spin  = xr.concat([ds_EBspin_fac_segs[0]['Espin_fac_y']], dim='time')
Ez_fac_spin  = xr.concat([ds_EBspin_fac_segs[0]['Espin_fac_z']], dim='time')
Bx_fac_spin  = xr.concat([ds_EBspin_fac_segs[0]['Bspin_fac_x']], dim='time')
By_fac_spin  = xr.concat([ds_EBspin_fac_segs[0]['Bspin_fac_y']], dim='time')
Bz_fac_spin  = xr.concat([ds_EBspin_fac_segs[0]['Bspin_fac_z']], dim='time')

S_para_spin  = (Ex_fac_spin * By_fac_spin - Ey_fac_spin * Bx_fac_spin) / mu_0 * 1E-12

S_para_toroidal_spin = Ex_fac_spin * By_fac_spin / mu_0 * 1E-12
S_para_poloidal_spin = - Ey_fac_spin * Bx_fac_spin / mu_0 * 1E-12

print(S_para_spin)
print(S_para_toroidal_spin)
print(S_para_poloidal_spin)

In [ ]:
import matplotlib as mpl

import matplotlib.pyplot as plt

mpl.rcParams['font.size'] = 15

# S_para_spin をプロット
fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(S_para_spin.time, np.abs(S_para_spin.values*1E3), s=1, c='k')
ax.set_yscale('log')
ax.set_xlabel('Time')
ax.set_ylabel(r'$|S_{\parallel}|$ (spin) [mW/m$^2$]')

ax.minorticks_on()
ax.grid(which='both', alpha=0.5)
ax.axhline(y=3E-3, c='r', linestyle='--', alpha=0.5)
ax.set_ylim(bottom=1E-6)

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'S_para_spin.png')
    print(fig_path)
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
Vph_toroidal    = np.abs(Ey_fac / Bx_fac) * 1E6 # [m/s]
Vph_poloidal    = np.abs(Ex_fac / By_fac) * 1E6 # [m/s]

Vph_perp_comp   = np.sqrt((Ex_fac**2E0 + Ey_fac**2E0) / (Bx_fac**2E0 + By_fac**2E0)) * 1E6 # [m/s]

# 22:30:45~22:32:45, 22:48:30~22:51:00, 23:05:45~23:08:15

In [ ]:
path_base_save_plot = (
    "/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330/wavelet_PSD"
)
os.makedirs(path_base_save_plot, exist_ok=True)

In [ ]:
#import matplotlib as mpl
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 20
#
#time_windows = [
#    (np.datetime64('2022-09-01T22:30:45'),
#     np.datetime64('2022-09-01T22:32:45')),
#
#    (np.datetime64('2022-09-01T22:48:30'),
#     np.datetime64('2022-09-01T22:51:00')),
#
#    (np.datetime64('2022-09-01T23:05:45'),
#     np.datetime64('2022-09-01T23:08:15')),
#]
#
#def add_panel_label(ax, label, x=-0.10, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#for t0, t1 in time_windows:
#    fig, axes = plt.subplots(len(targets_EB128_EBspin)+2, 1, figsize=(14, 18), sharex=True)
#
#    # --- dualband CWT 部分 ---
#    for ax, (v_hi, v_lo, ylab, unit) in zip(axes, targets_EB128_EBspin):
#        if (v_hi, v_lo) not in joined_EB128_EBspin_fac_cwt: continue
#        da_hi, coi_hi, da_lo, coi_lo = joined_EB128_EBspin_fac_cwt[(v_hi, v_lo)]
#        plot_cwt_dualband_on_ax(
#            ax, da_hi, coi_hi, da_lo, coi_lo,
#            t0=t0, t1=t1, minutes=None,
#            f_lo=(1e-2, 0.1824), f_hi=(0.1824, 64.0),
#            zrange=(1e-6, 1e3), cmap="turbo",
#            ylabel=ylab, unit_right=unit
#        )
#        ax.axhline(0.1824, c='k', lw=2)
#
#    S_para_window = S_para.sel(time=slice(t0, t1))
#    S_para_toroidal_window = S_para_toroidal.sel(time=slice(t0, t1))
#    S_para_poloidal_window = S_para_poloidal.sel(time=slice(t0, t1))
#
#    axes[len(targets_EB128_EBspin)].plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5, label='total')
#    axes[len(targets_EB128_EBspin)].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    axes[len(targets_EB128_EBspin)].minorticks_on()
#    axes[len(targets_EB128_EBspin)].grid(which='both', alpha=0.5)
#
##    axes[len(targets_EB128_EBspin)+1].plot(S_para_poloidal_window.time, S_para_poloidal_window.data*1E3, c='green', #linewidth=0.5, label='poloidal')
##    axes[len(targets_EB128_EBspin)+1].plot(S_para_toroidal_window.time, S_para_toroidal_window.data*1E3, c='orange', #linewidth=0.5, label='toroidal')
##    axes[len(targets_EB128_EBspin)+1].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
##    axes[len(targets_EB128_EBspin)+1].minorticks_on()
##    axes[len(targets_EB128_EBspin)+1].grid(which='both', alpha=0.5)
##    axes[len(targets_EB128_EBspin)+1].legend(ncol=5, loc='lower left', fontsize=10)
#
#    Vph_poloidal_window = Vph_poloidal.sel(time=slice(t0, t1))
#    Vph_toroidal_window = Vph_toroidal.sel(time=slice(t0, t1))
#    Vph_perp_window     = Vph_perp_comp.sel(time=slice(t0, t1))
#    VA_window = ds_velocity_ms_perp['Alfven_speed'].sel(time=slice(t0, t1))
#    print(Vph_perp_window)
#
#    axes[len(targets_EB128_EBspin)+1].plot(Vph_perp_window.time, Vph_perp_window.data*1E-3, c='k', lw=0.5, linestyle='solid', label=r'$|\mathbf{E}_{\perp}| / |\mathbf{B}_{\perp}|$')
#    axes[len(targets_EB128_EBspin)+1].plot(VA_window.time, VA_window.data*1E-3, c='blue', lw=2, linestyle='dotted', label=r'$v_{\mathrm{A}}$')
#    axes[len(targets_EB128_EBspin)+1].set_ylabel(r'Velocity' + '\n' + r'[$\mathrm{km/s}$]')
#    axes[len(targets_EB128_EBspin)+1].set_ylim(ymin=1E2, ymax=1E5)
#    axes[len(targets_EB128_EBspin)+1].set_yscale('log')
#    axes[len(targets_EB128_EBspin)+1].minorticks_on()
#    axes[len(targets_EB128_EBspin)+1].grid(which='both', alpha=0.5)
#    axes[len(targets_EB128_EBspin)+1].legend(ncol=2, loc='lower left', fontsize=15)
#
##    axes[len(targets_EB128_EBspin)+2].plot(Vph_poloidal_window.time, Vph_poloidal_window.data*1E-3, c='green', lw=0.5, ##linestyle='solid', label='poloidal')
##    axes[len(targets_EB128_EBspin)+2].plot(Vph_toroidal_window.time, Vph_toroidal_window.data*1E-3, c='orange', lw=0.5, ##linestyle='solid', label='toroidal')
##    axes[len(targets_EB128_EBspin)+2].plot(VA_window.time, VA_window.data*1E-3, c='blue', lw=2, linestyle='dotted', ##label=r'$v_{\mathrm{A}}$')
##    axes[len(targets_EB128_EBspin)+2].set_ylabel(r'Velocity' + '\n' + r'[$\mathrm{km/s}$]')
##    axes[len(targets_EB128_EBspin)+2].set_ylim(ymin=1E2, ymax=1E5)
##    axes[len(targets_EB128_EBspin)+2].set_yscale('log')
##    axes[len(targets_EB128_EBspin)+2].minorticks_on()
##    axes[len(targets_EB128_EBspin)+2].grid(which='both', alpha=0.5)
##    axes[len(targets_EB128_EBspin)+2].legend(ncol=5, loc='lower left', fontsize=10)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#    fig.subplots_adjust(hspace=0.1)
#
#    add_panel_label(axes[0], '(a)')
#    add_panel_label(axes[1], '(b)')
#    add_panel_label(axes[2], '(c)')
#    add_panel_label(axes[3], '(d)')
#    add_panel_label(axes[4], '(e)')
#    add_panel_label(axes[5], '(f)')
#    add_panel_label(axes[6], '(g)')
#    add_panel_label(axes[7], '(h)')
#    #add_panel_label(axes[8], '(9)')
#
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}_for_figure.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
targets_EB128_EBspin_analysis = [
    ("E128_fac_x_cwt","Espin_fac_x_cwt",  r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B128_fac_y_cwt","Bspin_fac_y_cwt",  r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
    ("E128_fac_y_cwt","Espin_fac_y_cwt",  r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B128_fac_x_cwt","Bspin_fac_x_cwt",  r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
]
joined_EB128_EBspin_fac_cwt_analysis = {}
for v_hi, v_lo, _, _ in targets_EB128_EBspin_analysis:
    da_hi, coi_hi = concat_cwt_segments(ds_EB128_fac_cwt_segs,  v_hi)
    da_lo, coi_lo = concat_cwt_segments(ds_EBspin_fac_cwt_segs, v_lo)
    if (da_hi is not None) or (da_lo is not None):
        joined_EB128_EBspin_fac_cwt_analysis[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#time_windows_analysis   = [np.datetime64('2022-09-01T22:25:00'), np.datetime64('2022-09-01T23:15:00')]
#
#ds_velocity_ms_poloidal_analysis    = ds_velocity_ms_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_toroidal_analysis    = ds_velocity_ms_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#S_para_toroidal_analysis    = S_para_toroidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#S_para_poloidal_analysis    = S_para_poloidal.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#S_para_toroidal_spin_analysis   = S_para_toroidal_spin.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#S_para_poloidal_spin_analysis   = S_para_poloidal_spin.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#import matplotlib.ticker as mticker
#from datetime import datetime
#import matplotlib.dates as mdates
#
#mpl.rcParams['font.size'] = 22
#
#fig = plt.figure(figsize=(12, 22))
#gs = fig.add_gridspec(10, 2, height_ratios=[1, 1, 0.52546, 0.75, 0.75, 1, 1, 0.52546, 0.75, 0.75], width_ratios=[1, 0.02], wspace=0.05, hspace=0.15)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)
#ax_8 = fig.add_subplot(gs[8, 0], sharex=ax_0)
#ax_9 = fig.add_subplot(gs[9, 0], sharex=ax_0)
#
#cax_0 = fig.add_subplot(gs[0, 1])
#cax_1 = fig.add_subplot(gs[1, 1])
#cax_2 = fig.add_subplot(gs[2, 1])
#cax_5 = fig.add_subplot(gs[5, 1])
#cax_6 = fig.add_subplot(gs[6, 1])
#cax_7 = fig.add_subplot(gs[7, 1])
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#ax_6.tick_params(axis='x', which='both', labelbottom=False)
#ax_7.tick_params(axis='x', which='both', labelbottom=False)
#ax_8.tick_params(axis='x', which='both', labelbottom=False)
#
#axes = [ax_0, ax_1, ax_5, ax_6]
#caxes = [cax_0, cax_1, cax_5, cax_6]
#for ax, cax, (v_hi, v_lo, ylab, unit) in zip(axes, caxes, targets_EB128_EBspin_analysis):
#    if (v_hi, v_lo) not in joined_EB128_EBspin_fac_cwt_analysis: continue
#    da_hi, coi_hi, da_lo, coi_lo = joined_EB128_EBspin_fac_cwt_analysis[(v_hi, v_lo)]
#    plot_cwt_prefer_hi_fill_lo(
#        ax,
#        da_hi, coi_hi,
#        da_lo, coi_lo,
#        t0=time_windows_analysis[0], t1=time_windows_analysis[1], minutes=None,
#        f_range=(1e-2, 64.),
#        zrange=(1e-6, 1e3),
#        cmap="turbo",
#        ylabel=ylab, unit_right=unit, cax=cax
#    )
#
#axes_phi = [ax_2, ax_7]
#caxes_phi = [cax_2, cax_7]
#for i, (w_var, p_var, lab) in enumerate(targets_EB128_xwt):
#    if lab not in joined_EB128_xwt: continue
#    w_da, p_da = joined_EB128_xwt[lab]
#    plot_xwt_phase_on_ax(axes_phi[i], w_da, p_da, t0=time_windows_analysis[0], minutes=50, mode="phase",
#                         yrange=(1e-2, 1e0), unit='[deg]', cax=caxes_phi[i])
#
#
## ax_2はPhase (abs) Ex-By (128 Hzのみ)
#
#ax_3.scatter(S_para_toroidal_analysis.time, S_para_toroidal_analysis.data*1E3, c='k', s=0.1, label='128 Hz')
#ax_3.plot(S_para_toroidal_spin_analysis.time, S_para_toroidal_spin_analysis.data*1E3, c='b', lw=1, label='Spin')
#ax_4.plot(ds_velocity_ms_toroidal_analysis.time, ds_velocity_ms_toroidal_analysis.perp_ion_speed * 1E-3, lw=1, c='k')
#ax_4.axhline(0., c='gray', lw=2, alpha=0.5, linestyle='dashed')
#
## ax_7はPhase (abs) Ey-Bx (128 Hzのみ)
#
#ax_8.scatter(S_para_poloidal_analysis.time, S_para_poloidal_analysis.data*1E3, c='k', s=0.1, label='128 Hz')
#ax_8.plot(S_para_poloidal_spin_analysis.time, S_para_poloidal_spin_analysis.data*1E3, c='b', lw=1, label='Spin')
#ax_9.plot(ds_velocity_ms_poloidal_analysis.time, ds_velocity_ms_poloidal_analysis.perp_ion_speed * 1E-3, lw=1, c='k')
#ax_9.axhline(0., c='gray', lw=2, alpha=0.5, linestyle='dashed')
#
#
#ax_0.set_ylabel(r'$E_{x}$'                      + '\n' + r'[$\mathrm{Hz}$]')
#ax_1.set_ylabel(r'$B_{y}$'                      + '\n' + r'[$\mathrm{Hz}$]')
#ax_2.set_ylabel(r'$|\phi_{\mathrm{tor}}|$'      + '\n' + r'[$\mathrm{Hz}$]')
#ax_3.set_ylabel(r'$S_{\parallel \mathrm{tor}}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#ax_4.set_ylabel(r'$V_{\mathrm{ion} x}$'         + '\n' + r'[$\mathrm{km/s}$]')
#
#ax_5.set_ylabel(r'$E_{y}$'                      + '\n' + r'[$\mathrm{Hz}$]')
#ax_6.set_ylabel(r'$B_{x}$'                      + '\n' + r'[$\mathrm{Hz}$]')
#ax_7.set_ylabel(r'$|\phi_{\mathrm{pol}}|$'      + '\n' + r'[$\mathrm{Hz}$]')
#ax_8.set_ylabel(r'$S_{\parallel \mathrm{pol}}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#ax_9.set_ylabel(r'$V_{\mathrm{ion} y}$'         + '\n' + r'[$\mathrm{km/s}$]')
#
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#ax_7.minorticks_on()
#ax_7.grid(which='both', alpha=0.5)
#ax_8.minorticks_on()
#ax_8.grid(which='both', alpha=0.5)
#ax_9.minorticks_on()
#ax_9.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_9.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#ax_9.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
#ax_9.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))
#
#def add_panel_label(ax, label, x=-0.15, y=0.90):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#def to_py_datetime(t_np64):
#    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)
#
## 軌道データ
#t_pos_py = to_py_datetime(THA_SM_pos.time.values)
#t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数
#
#R   = np.asarray(THA_rmlatmlt_R, dtype=float)  # Re
#mlat= np.asarray(THA_rmlatmlt_MLAT, dtype=float)  # deg
#mlt = np.asarray(THA_rmlatmlt_MLT, dtype=float)  # hour [0,24)
#
#L_shell = np.asarray(THA_rmlatmlt_L, dtype=float)
#
## --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
#mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)
#
## 補間関数（tick の x は「日数」なのでそのまま使う）
#def interp_at(x_num):
#    #Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
#    Ri    = np.interp(x_num, t_pos_num, L_shell, left=np.nan, right=np.nan)   # L-shell
#    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
#    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
#    mlti  = np.mod(mltiu, 24.0)
#    return Ri, mlati, mlti
#
## 目盛フォーマッタ
#def rmlt_formatter(x, pos=None):
#    Ri, mlati, mlti = interp_at(x)
#    if np.any(~np.isfinite([Ri, mlati, mlti])):
#        return ""  # 範囲外は空
#    return (f"{Ri:0.2f}\n"
#            f"{mlati:0.2f}\n"
#            f"{mlti:0.2f}")
#
## セカンダリ x 軸（底 side）を作ってラベルを差し替え
#secax = ax_9.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
#secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))
#
## メインの時間ラベルと重ならないよう余白を広げる
#ax_9.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
#secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル
#
#secax.set_ticks(ax_9.get_xticks())
#xlab = 0.08
#fig.text(xlab, 0.062, "hhmm", ha='center', va='center')
#fig.text(xlab, 0.044, "L-shell", ha='center', va='center')
#fig.text(xlab, 0.028, "MLAT", ha='center', va='center')
#fig.text(xlab, 0.012, "MLT", ha='center', va='center')
#
#add_panel_label(ax_0, '(a)')
#add_panel_label(ax_1, '(b)')
#add_panel_label(ax_2, '(c)')
#add_panel_label(ax_3, '(d)')
#add_panel_label(ax_4, '(e)')
#add_panel_label(ax_5, '(f)')
#add_panel_label(ax_6, '(g)')
#add_panel_label(ax_7, '(h)')
#add_panel_label(ax_8, '(i)')
#add_panel_label(ax_9, '(j)')
#
#fig.suptitle('THEMIS-A', y=0.995)   # タイトルは上に寄せる
#
#fig.subplots_adjust(
#    left=0.18,
#    right=0.88,
#    top=0.975,
#    bottom=0.07,
#    hspace=0.15,
#    wspace=0.05,
#)
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'Figure_3_a.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    #fig.savefig(os.path.join(path_base_save_plot, 'Figure_3_a.pdf'))
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# E/B plot

- Poloidal components: $B_{x}$, $E_{y}$
- Toroidal components: $B_{y}$, $E_{x}$

In [ ]:
ds_EBspin_fac_toroidal  = xr.Dataset({
    'Espin':        joined_EBspin_fac_cwt['Espin_fac_x_cwt'][0],
    'Bspin':        joined_EBspin_fac_cwt['Bspin_fac_y_cwt'][0]
})

ds_EBspin_fac_poloidal  = xr.Dataset({
    'Espin':        joined_EBspin_fac_cwt['Espin_fac_y_cwt'][0],
    'Bspin':        joined_EBspin_fac_cwt['Bspin_fac_x_cwt'][0]
})

print(ds_EBspin_fac_toroidal)
print(ds_EBspin_fac_poloidal)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

# 元の時間範囲
t0 = pd.Timestamp(ds_EBspin_fac_toroidal.time.min().values)
t1 = pd.Timestamp(ds_EBspin_fac_toroidal.time.max().values)

# 「毎秒 +0.5秒」のグリッドを作る
# まず t0 の属する秒の 0.500 に揃える（t0 がそれより後なら次の秒へ）
anchor = t0.floor("S") + pd.Timedelta(milliseconds=500)
if anchor < t0:
    anchor = anchor + pd.Timedelta(seconds=1)

# 1秒刻みで t1 まで（範囲内に限定）
new_time = pd.date_range(start=anchor, end=t1, freq="1S")

# 線形補間（time方向のみ）
ds_EBspin_fac_toroidal_interp = ds_EBspin_fac_toroidal.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

ds_EBspin_fac_poloidal_interp = ds_EBspin_fac_poloidal.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

ds_parameter_interp = ds_parameter.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

ds_velocity_ms_toroidal_interp  = ds_velocity_ms_toroidal.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

ds_velocity_ms_poloidal_interp  = ds_velocity_ms_poloidal.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

da_Spara_toroidal_spin_interp        = S_para_toroidal_spin.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

da_Spara_poloidal_spin_interp        = S_para_poloidal_spin.interp(
    time=xr.DataArray(new_time, dims="time"),
    method="linear"
)

print(ds_EBspin_fac_toroidal_interp)
print(ds_EBspin_fac_poloidal_interp)
print(ds_parameter_interp)
print(ds_velocity_ms_toroidal_interp)
print(ds_velocity_ms_poloidal_interp)

In [ ]:
print(np.nanmax(np.abs(da_Spara_toroidal_spin_interp))*1E3)

print(np.nanmax(np.abs(S_para_toroidal_spin))*1E3)

In [ ]:
ds_wco_0 = ds_EB128_fac_cwt_segs[0]['wco_sig95'].broadcast_like(ds_EB128_fac_cwt_segs[0]['EB128_wco_exby'])
ds_wco_1 = ds_EB128_fac_cwt_segs[1]['wco_sig95'].broadcast_like(ds_EB128_fac_cwt_segs[1]['EB128_wco_exby'])
ds_wco_2 = ds_EB128_fac_cwt_segs[2]['wco_sig95'].broadcast_like(ds_EB128_fac_cwt_segs[2]['EB128_wco_exby'])

In [ ]:
ds_EB128_fac_toroidal  = xr.Dataset({
    'E128':         xr.concat([ds_EB128_fac_cwt_segs[0]['E128_fac_x_cwt'], ds_EB128_fac_cwt_segs[1]['E128_fac_x_cwt'], ds_EB128_fac_cwt_segs[2]['E128_fac_x_cwt']], dim='time'),
    'B128':         xr.concat([ds_EB128_fac_cwt_segs[0]['B128_fac_y_cwt'], ds_EB128_fac_cwt_segs[1]['B128_fac_y_cwt'], ds_EB128_fac_cwt_segs[2]['B128_fac_y_cwt']], dim='time'),
    'coherency':    xr.concat([ds_EB128_fac_cwt_segs[0]['EB128_wco_exby'], ds_EB128_fac_cwt_segs[1]['EB128_wco_exby'], ds_EB128_fac_cwt_segs[2]['EB128_wco_exby']], dim='time'),
    'phase':        xr.concat([ds_EB128_fac_cwt_segs[0]['EB128_phase_exby'], ds_EB128_fac_cwt_segs[1]['EB128_phase_exby'], ds_EB128_fac_cwt_segs[2]['EB128_phase_exby']], dim='time'),
    'wco_sig95':    xr.concat([ds_wco_0, ds_wco_1, ds_wco_2], dim='time')
})

print(ds_EB128_fac_toroidal)

ds_EB128_fac_poloidal  = xr.Dataset({
    'E128':         xr.concat([ds_EB128_fac_cwt_segs[0]['E128_fac_y_cwt'], ds_EB128_fac_cwt_segs[1]['E128_fac_y_cwt'], ds_EB128_fac_cwt_segs[2]['E128_fac_y_cwt']], dim='time'),
    'B128':         xr.concat([ds_EB128_fac_cwt_segs[0]['B128_fac_x_cwt'], ds_EB128_fac_cwt_segs[1]['B128_fac_x_cwt'], ds_EB128_fac_cwt_segs[2]['B128_fac_x_cwt']], dim='time'),
    'coherency':    xr.concat([ds_EB128_fac_cwt_segs[0]['EB128_wco_eybx'], ds_EB128_fac_cwt_segs[1]['EB128_wco_eybx'], ds_EB128_fac_cwt_segs[2]['EB128_wco_eybx']], dim='time'),
    'phase':        xr.concat([ds_EB128_fac_cwt_segs[0]['EB128_phase_eybx'], ds_EB128_fac_cwt_segs[1]['EB128_phase_eybx'], ds_EB128_fac_cwt_segs[2]['EB128_phase_eybx']], dim='time'),
    'wco_sig95':    xr.concat([ds_wco_0, ds_wco_1, ds_wco_2], dim='time')
})

print(ds_EB128_fac_poloidal)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

# 元の時間範囲
t0 = pd.Timestamp(ds_EBspin_fac_toroidal.time.min().values)
t1 = pd.Timestamp(ds_EBspin_fac_toroidal.time.max().values)

# 「毎秒 +0.5秒」のグリッドを作る
# まず t0 の属する秒の 0.500 に揃える（t0 がそれより後なら次の秒へ）
anchor = t0.floor("S") + pd.Timedelta(milliseconds=500)
if anchor < t0:
    anchor = anchor + pd.Timedelta(seconds=1)

# 1秒刻みで t1 まで（範囲内に限定）
new_time = pd.date_range(start=anchor, end=t1, freq="1S")

ds_EB128_fac_toroidal_avg = (
    ds_EB128_fac_toroidal
    .resample(time="1S", offset="500ms")
    .mean()
)

ds_EB128_fac_poloidal_avg = (
    ds_EB128_fac_poloidal
    .resample(time="1S", offset="500ms")
    .mean()
)

ds_EB128_fac_toroidal_avg = ds_EB128_fac_toroidal_avg.reindex(
    time=new_time
)

ds_EB128_fac_poloidal_avg = ds_EB128_fac_poloidal_avg.reindex(
    time=new_time
)

da_Spara_toroidal_128_avg       = (
    S_para_toroidal
    .resample(time="1S", offset="500ms")
    .mean()
)

da_Spara_poloidal_128_avg       = (
    S_para_poloidal
    .resample(time="1S", offset="500ms")
    .mean()
)
print(da_Spara_toroidal_128_avg)
print(da_Spara_poloidal_128_avg)

da_Spara_toroidal_128_avg       = da_Spara_toroidal_128_avg.reindex(
    time=new_time
)

da_Spara_poloidal_128_avg       = da_Spara_poloidal_128_avg.reindex(
    time=new_time
)

print(ds_EB128_fac_toroidal_avg)
print(ds_EB128_fac_poloidal_avg)
print(da_Spara_toroidal_128_avg)
print(da_Spara_poloidal_128_avg)

In [ ]:
print(np.nanmax(np.abs(da_Spara_toroidal_128_avg))*1E3)

print(np.nanmax(np.abs(S_para_toroidal))*1E3)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

def _fit_powerlaw_kappa(freq, psd, mask, min_points=5):

    f = np.asarray(freq)
    y = np.asarray(psd)

    msk = np.asarray(mask, dtype=bool)
    msk &= np.isfinite(f) & np.isfinite(y) & (f > 0) & (y > 0)

    if msk.sum() < min_points:
        return np.nan, np.nan

    x = np.log10(f[msk])
    yy = np.log10(y[msk])

    N = len(x)

    # 線形回帰
    slope, intercept = np.polyfit(x, yy, 1)

    # 残差
    y_fit = intercept + slope * x
    residual = yy - y_fit

    # 残差分散
    sigma2 = np.sum(residual**2) / (N - 2)

    Sxx = np.sum((x - x.mean())**2)

    if Sxx == 0:
        return np.nan, np.nan

    slope_std = np.sqrt(sigma2 / Sxx)

    kappa = -slope
    kappa_std = slope_std

    return kappa, kappa_std


def _corr_log_model(freq, y_obs, y_model_at_freq, min_points=5):
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model_at_freq)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    if msk.sum() < min_points:
        return np.nan, int(msk.sum())

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    # 分散が小さいと相関が不安定なので弾く
    if np.std(logy) < 1e-6 or np.std(logm) < 1e-6:
        return np.nan, int(msk.sum())

    r = np.corrcoef(logy, logm)[0, 1]
    return float(r), int(msk.sum())

def _logrmse_model(freq, y_obs, y_model, min_points=5, allow_offset=False):
    """
    logRMSE = sqrt(mean((log10(y_obs) - (log10(y_model)+a))^2))
    allow_offset=True: a を平均差で最小二乗フィット（縦オフセット許容）
    """
    f = np.asarray(freq)
    y = np.asarray(y_obs)
    m = np.asarray(y_model)

    msk = np.isfinite(f) & np.isfinite(y) & np.isfinite(m) & (f > 0) & (y > 0) & (m > 0)

    n = int(msk.sum())
    if n < min_points:
        return np.nan, np.nan, n  # (logRMSE, a, n)

    logy = np.log10(y[msk])
    logm = np.log10(m[msk])

    a = 0.0
    if allow_offset:
        a = float(np.mean(logy - logm))
        resid = logy - (logm + a)
    else:
        resid = logy - logm

    rmse = float(np.sqrt(np.mean(resid**2)))
    return rmse, a, n



def plot_freq_spectrum(time, ds_spin, ds_128, ds_par, ds_vel, da_Spara_spin, da_Spara_128, title_label,
                      E2_label, B2_label, phi_label, EBratio_label, vsys_label, S_para_label,
                      fit_range=[3.0, 30.0]):

    time    = pd.Timestamp(time)
    time_0  = time - np.timedelta64(500, 'ms')
    time_1  = time + np.timedelta64(500, 'ms')

    ds_spin_time = ds_spin.sel(time=time, method="nearest")
    ds_128_time  = ds_128.sel(time=time, method="nearest")
    ds_par_time  = ds_par.sel(time=time, method="nearest")
    ds_vel_time  = ds_vel.sel(time=time, method="nearest")
    da_Spara_spin_time  = da_Spara_spin.sel(time=time, method="nearest")
    da_Spara_128_time   = da_Spara_128.sel(time=time, method="nearest")

    # 理論曲線
    f_sc    = np.logspace(-2, 2, 1000)
    tau     = ds_par_time['i-e_temp_ratio'].item()
    f_ci    = ds_par_time['proton_cycl_freq_Hz'].item()
    v_thi   = ds_vel_time['ion_thermal_speed'].item()
    v_sys   = ds_vel_time['perp_sys_speed'].item()
    Spara_spin  = da_Spara_spin_time.item()
    Spara_128   = da_Spara_128_time.item()
    if np.isnan(Spara_128):
        Spara = Spara_spin
    else:
        Spara = Spara_128
    KAW_dr  = (1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (f_sc / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))

    if fit_range[1] == 0:
        fit_range[1] = np.sqrt(1.67262192E-27 / 9.1093837E-31 / tau)

    f_sc_krho_1     = np.abs(f_ci * v_sys / v_thi)
    f_sc_krho_low   = f_sc_krho_1 * fit_range[0]
    f_sc_krho_high  = f_sc_krho_1 * fit_range[1]

    f_spin_1 = 0.3648
    f_spin_2 = f_spin_1 * 2.
    f_spin_3 = f_spin_1 * 3.
    f_spin_4 = f_spin_1 * 4.
    f_spin_5 = f_spin_1 * 5.

    # PSD
    E_spin  = ds_spin_time['Espin']
    B_spin  = ds_spin_time['Bspin']

    E_128   = ds_128_time['E128']
    B_128   = ds_128_time['B128']

    coherency   = ds_128_time['coherency']
    wco_sig95   = ds_128_time['wco_sig95']
    phase       = ds_128_time['phase']

    v_A     = ds_vel_time['Alfven_speed'].item()

    EB_spin_ratio   = np.sqrt(E_spin / B_spin) * 1E6 / v_A
    EB_128_ratio    = np.sqrt(E_128 / B_128) * 1E6 / v_A
    EB_128_ratio = EB_128_ratio.where(coherency >= wco_sig95)

    # --- fit用マスク（coherency + fit_range） ---
    freq = E_128.freq.values
    mask_coh = (coherency >= wco_sig95).values
    mask_fit = (freq >= f_sc_krho_low) & (freq <= f_sc_krho_high) & (freq >= 1E-2)
    mask_all = mask_coh & mask_fit

    KAW_dr_corr = (1. + (freq / f_ci * v_thi / v_sys)**2 / 2.) / np.sqrt(1. + (freq / f_ci * v_thi / v_sys)**2 / 2. * (1. + 1. / tau))
    KAW_dr_corr = KAW_dr_corr[mask_all]
    EB_128_ratio_corr = EB_128_ratio[mask_all]

    r_EB, n_EB              = _corr_log_model(EB_128_ratio_corr.freq, EB_128_ratio_corr.data, KAW_dr_corr, min_points=30)
    logrmse_EB_abs, _, _    = _logrmse_model(EB_128_ratio_corr.freq, EB_128_ratio_corr.data, KAW_dr_corr, min_points=30)

    # κ推定（失敗なら NaN）
    kappa_E, kappa_E_err    = _fit_powerlaw_kappa(freq, E_128.values, mask_all, min_points=30)
    kappa_B, kappa_B_err    = _fit_powerlaw_kappa(freq, B_128.values, mask_all, min_points=30)

    # plot
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    mpl.rcParams['font.size'] = 20

    fig     = plt.figure(figsize=(10, 15))
    gs      = fig.add_gridspec(4, 1, height_ratios=[4, 3, 3, 4], hspace=0.15)
    ax_0    = fig.add_subplot(gs[0, 0])
    ax_1    = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2    = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3    = fig.add_subplot(gs[3, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)

    # 灰色マスク（coherency>=sig95 だけ。fit_rangeではなく“灰色領域”を強調したいならこれ）
    mask_gray = mask_coh
    ax_0.plot(E_spin.freq,  E_spin, lw=2, linestyle='dotted', c='green')
    ax_0.plot(B_spin.freq,  B_spin, lw=2, linestyle='dotted', c='purple')
    ax_0.plot(E_128.freq,   E_128,  lw=2, linestyle='solid',  c='green')
    ax_0.plot(B_128.freq,   B_128,  lw=2, linestyle='solid',  c='purple')
    ax_0.set_yscale('log')
    ax_0.set_ylim(1E-6, 1E4)
    ax_0.set_yticks([1E-6, 1E-5, 1E-4, 1E-3, 1E-2, 1E-1, 1E0, 1E1, 1E2, 1E3, 1E4])
    ax_0.set_xscale('log')
    ax_0.set_xlim(1E-2, 64)
    ax_0.minorticks_on()
    ax_0.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_0.fill_between(freq, 1E-6, 1E4, where=mask_gray, color='gray', alpha=0.30)

    if np.isfinite(kappa_E):
        txt_E = rf'$\kappa_E$={kappa_E:.2f}±{kappa_E_err:.2f}'
        ax_0.text(
            0.98, 0.95, txt_E,
            transform=ax_0.transAxes,
            ha='right', va='top',
            color='green',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )

    if np.isfinite(kappa_B):
        txt_B = rf'$\kappa_B$={kappa_B:.2f}±{kappa_B_err:.2f}'
        ax_0.text(
            0.98, 0.82, txt_B,
            transform=ax_0.transAxes,
            ha='right', va='top',
            color='purple',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )

    ax_0.set_ylabel(E2_label + r' [$\mathrm{(mV/m)^{2}/Hz}$]' + '\n' + B2_label + r' [$\mathrm{nT^{2}/Hz}$]')

    ax_1.plot(wco_sig95.freq, wco_sig95, lw=2, linestyle='dotted', c='r')
    ax_1.plot(coherency.freq, coherency, lw=2, linestyle='solid',  c='k')
    ax_1.set_yscale('log')
    ax_1.set_ylim(1E-3, 1)
    ax_1.minorticks_on()
    ax_1.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_1.set_ylabel('Coherency')

    ax_2.plot(phase.freq, np.abs(phase), lw=2, linestyle='solid', c='k')
    ax_2.axhline(90, lw=2, linestyle='dashed', c='gray', alpha=0.5)
    ax_2.set_ylim(0, 180)
    ax_2.set_yticks([0, 45, 90, 135, 180])
    ax_2.minorticks_on()
    ax_2.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_2.set_ylabel('|' + phi_label + '| [deg]')

    ax_3.plot(EB_spin_ratio.freq, EB_spin_ratio, lw=2, linestyle='dotted', c='k')
    ax_3.plot(EB_128_ratio.freq,  EB_128_ratio,  lw=2, linestyle='solid',  c='k')
    ax_3.plot(f_sc, KAW_dr, lw=2, linestyle='dotted', c='r')
    ax_3.set_yscale('log')
    ax_3.set_ylim(1E-1, 1E3)
    ax_3.minorticks_on()
    ax_3.grid(which='both', alpha=0.3, linestyle='dashed')
    ax_3.set_ylabel(EBratio_label)
    ax_3.set_xlabel('Frequency [Hz]')
    if np.isfinite(r_EB):
        ax_3.text(
            0.98, 0.34, r'$r_{\mathrm{EB}}$ =' + f'{r_EB:.2f}',
            transform=ax_3.transAxes, ha='right', va='center',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )
    if np.isfinite(logrmse_EB_abs):
        ax_3.text(
            0.98, 0.21, rf'logRMSE={logrmse_EB_abs:.2f}',
            transform=ax_3.transAxes, ha='right', va='center',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )
    ax_3.text(0.98, 0.08, vsys_label + f'={v_sys*1E-3:.2f} [km/s]',
              transform=ax_3.transAxes, ha='right', va='center',
              bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2))

    ax_0.set_title(
        f"{time_0.strftime('%H:%M:%S.%f')} - {time_1.strftime('%H:%M:%S.%f')}" + '\n' +
        title_label + ', ' + S_para_label + f' = {(Spara*1E3):.3f} ' + r'[$\mathrm{mW/m^{2}}$]'
    )

    for ax in [ax_0, ax_1, ax_2, ax_3]:
        ax.axvline(f_spin_1, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_2, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_3, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_4, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_spin_5, lw=2, linestyle='dotted', c='blue', alpha=0.5)
        ax.axvline(f_sc_krho_1,   lw=2, linestyle='dashed', c='orange',  alpha=0.7)
        ax.axvline(f_sc_krho_low, lw=2, linestyle='dashed', c='magenta', alpha=0.7)
        ax.axvline(f_sc_krho_high,lw=2, linestyle='dashed', c='magenta', alpha=0.7)

    fig.tight_layout()

    fit_results = {
        'time':         time,
        'kappa_E':      kappa_E if kappa_E else np.nan,
        'kappa_E_err':  kappa_E_err if kappa_E_err else np.nan,
        'kappa_B':      kappa_B if kappa_B else np.nan,
        'kappa_B_err':  kappa_B_err if kappa_B_err else np.nan,
        'r_EB':         r_EB if r_EB else np.nan,
        'logrmse_EB':   logrmse_EB_abs if logrmse_EB_abs else np.nan,
        'n_EB':         n_EB if n_EB else np.nan
    }

    return fig, fit_results

In [ ]:
plot_freq_spectrum('2022-09-01T22:32:08.500000', ds_EBspin_fac_toroidal_interp, ds_EB128_fac_toroidal_avg, ds_parameter_interp, ds_velocity_ms_toroidal_interp, da_Spara_toroidal_spin_interp, da_Spara_toroidal_128_avg, r'toroidal', r'$E_{x}^{2}$', r'$B_{y}^{2}$', r'$\phi_{\mathrm{tor}}$', r'$\sqrt{E_{x}^{2} / B_{y}^{2}} / v_{\mathrm{A}}$', r'$v_{\mathrm{sys}x}$', r'$S_{\parallel\mathrm{tor}}$', fit_range=[3, 100])

In [ ]:
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed

class TqdmJoblib(tqdm):
    """
    joblib.Parallel の進捗を「完了ベース」で tqdm に反映するコンテキストマネージャ
    """
    def __enter__(self):
        self._old_cb = joblib.parallel.BatchCompletionCallBack
        pbar = self

        class _BatchCompletionCallBack(self._old_cb):
            def __call__(self, *args, **kwargs):
                # 完了したバッチサイズ分だけ進捗を進める
                pbar.update(n=self.batch_size)
                return super().__call__(*args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = _BatchCompletionCallBack
        return super().__enter__()

    def __exit__(self, exc_type, exc, tb):
        joblib.parallel.BatchCompletionCallBack = self._old_cb
        return super().__exit__(exc_type, exc, tb)

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
#direction           = 'poloidal'
#E2_label            = r'$E_{y}^{2}$'
#B2_label            = r'$B_{x}^{2}$'
#phi_label           = r'$\phi_{\mathrm{pol}}$'
#EBratio_label       = r'$\sqrt{E_{y}^{2} / B_{x}^{2}} / v_{\mathrm{A}}$'
#vsys_label          = r'$v_{\mathrm{sys}y}$'
#Spara_label         = r'$S_{\parallel\mathrm{pol}}$'
#
#ds_spin             = ds_EBspin_fac_poloidal_interp
#ds_128              = ds_EB128_fac_poloidal_avg
#ds_par              = ds_parameter_interp
#ds_vel              = ds_velocity_ms_poloidal_interp
#da_Spara_spin       = da_Spara_poloidal_spin_interp
#da_Spara_128        = da_Spara_poloidal_128_avg
#
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/EB_ratio_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
#out_dir_pdf = f'{out_dir}/PDF/'
#os.makedirs(out_dir_pdf, exist_ok=True)
#out_dir_png = f'{out_dir}/PNG/'
#os.makedirs(out_dir_png, exist_ok=True)
#
#def process_and_save_plot(t):
#    fig, fit_results = plot_freq_spectrum(t, ds_spin, ds_128, ds_par, ds_vel, da_Spara_spin, da_Spara_128,
#                                          direction, E2_label, B2_label, phi_label,
#                                          EBratio_label, vsys_label, Spara_label, fit_range=[3, 0]) # np.sqrt(1.67262192E-27 / 9.1093837E-31)
#    if fig is None:
#        return None
#
#    ts = pd.Timestamp(t)
#    base = ts.strftime('%Y-%m-%dT%H%M%S%f')
#    png_path = os.path.join(out_dir_png, base + '.png')
#    pdf_path = os.path.join(out_dir_pdf, base + '.pdf')
#
#    try:
#        fig.savefig(png_path, dpi=200, bbox_inches='tight')
#        fig.savefig(pdf_path, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#    return fit_results
#
#time_range = ['2022-09-01T20:50:00', '2022-09-02T00:00:00']
#
#t_min, t_max    = pd.to_datetime(time_range)
#time_grid = ds_spin.sel(time=slice(t_min, t_max)).time.values
#
#with TqdmJoblib(total=len(time_grid), desc="Saving plots", unit="frame") as pbar:
#    results_list = Parallel(n_jobs=-1, backend="loky", verbose=0)(
#        delayed(process_and_save_plot)(t) for t in time_grid
#    )
#print('Finished saving all plots!')
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['font.size'] = 20

direction           = 'toroidal'

ds_spin             = ds_EBspin_fac_toroidal_interp
ds_128              = ds_EB128_fac_toroidal_avg
ds_par              = ds_parameter_interp
ds_vel              = ds_velocity_ms_toroidal_interp
S_par_spin          = da_Spara_toroidal_spin_interp
S_par_128           = da_Spara_toroidal_128_avg

S_par_label         = r'$S_{\parallel\mathrm{tor}}$'
v_sys_label         = r'$v_{\mathrm{sys}x}$'

out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/EB_ratio_{direction}'

time_range_data     = ['2022-09-01T20:50:00', '2022-09-02T00:00:00']
t_min_data, t_max_data    = pd.to_datetime(time_range_data)
time_str_start_data = t_min_data.strftime('%Y%m%d_%H%M%S')
time_str_end_data = t_max_data.strftime('%Y%m%d_%H%M%S')

kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start_data}_to_{time_str_end_data}.csv'
csv_path = os.path.join(out_dir, kappa_csv_filename)

#time_range_analysis = ['2022-09-01T22:28:50', '2022-09-01T22:34:29']
#time_range_analysis = ['2022-09-01T22:46:02', '2022-09-01T22:57:25']
#time_range_analysis = ['2022-09-01T23:03:50', '2022-09-01T23:09:29']
time_range_analysis = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
t_min, t_max = pd.to_datetime(time_range_analysis)
time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
time_str_end = t_max.strftime('%Y%m%d_%H%M%S')

data = pd.read_csv(csv_path)

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time")
data = data.set_index("time").to_xarray()
data = data.sel(time=slice(t_min, t_max))

S_par_spin_window   = S_par_spin.sel(time=slice(t_min, t_max))
S_par_128_window    = S_par_128.sel(time=slice(t_min, t_max))
v_sys_window        = ds_vel['perp_sys_speed'].sel(time=slice(t_min, t_max))

data, S_par_spin_window, S_par_128_window, v_sys_window = xr.align(
    data, S_par_spin_window, S_par_128_window, v_sys_window, join='inner'
)
print(np.nanmax(S_par_128_window))

m = (
    (data['r_EB'].data >= 0.4) &
    (data['logrmse_EB'].data <= 0.5) &
    (data['n_EB'].data >= 50) &
    (np.abs(S_par_128_window.data * 1E3) >= 3E-3)
)

x = pd.to_datetime(data['time'].values[m])
yB = data['kappa_B'].values[m]
eB = data['kappa_B_err'].values[m]
yE = data['kappa_E'].values[m]
eE = data['kappa_E_err'].values[m]

yB_valid = yB[np.isfinite(yB)]
yE_valid = yE[np.isfinite(yE)]

# 平均・標準偏差（サンプル標準偏差 ddof=1 推奨。母集団なら ddof=0）
if yB_valid.size > 0:
    kB_mean = float(np.mean(yB_valid))
    kB_std  = float(np.std(yB_valid, ddof=1)) if yB_valid.size > 1 else 0.0
else:
    kB_mean, kB_std = np.nan, np.nan

if yE_valid.size > 0:
    kE_mean = float(np.mean(yE_valid))
    kE_std  = float(np.std(yE_valid, ddof=1)) if yE_valid.size > 1 else 0.0
else:
    kE_mean, kE_std = np.nan, np.nan

# --- 時間変化をプロット ---
fig_kappa = plt.figure(figsize=(15, 8))
gs = fig_kappa.add_gridspec(4, 1)
ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
ax_2 = fig_kappa.add_subplot(gs[3, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)

# kappa_Bのプロット（エラーバー付き）
ax_0.errorbar(x, yB, yerr=eB, fmt='o', linestyle='none',
              color='purple', label=r'$\kappa_{\mathrm{B}}$', ms=4, capsize=3, elinewidth=1)
ax_0.axhline(7./3., lw=2, linestyle='dotted', c='purple', label=r'$\kappa_{\mathrm{B}}$ in KAW cascade')
if np.isfinite(kB_mean):
    ax_0.axhline(kB_mean, lw=2, linestyle='--', c='purple', label=rf'$\kappa_{{\mathrm{{B}}}}$ mean ({kB_mean:.2f})')

# kappa_Eのプロット（エラーバー付き）
ax_0.errorbar(x, yE, yerr=eE, fmt='o', linestyle='none',
              color='green', label=r'$\kappa_{\mathrm{E}}$', ms=4, capsize=3, elinewidth=1)
ax_0.axhline(1./3., lw=2, linestyle='dotted', c='green', label=r'$\kappa_{\mathrm{E}}$ in KAW cascade')
if np.isfinite(kE_mean):
    ax_0.axhline(kE_mean, lw=2, linestyle='--', c='green', label=rf'$\kappa_{{\mathrm{{E}}}}$ mean ({kE_mean:.2f})')

ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)' + f' ({direction})' + '\n' + rf'$\kappa_{{\mathrm{{E}}}} = {kE_mean:.2f} \pm {kE_std:.2f}$, $\kappa_{{\mathrm{{B}}}} = {kB_mean:.2f} \pm {kB_std:.2f}$')
ax_0.set_ylabel(r'Spectral index $\kappa$')
#ax_0.legend(ncol=1)
ax_0.minorticks_on()
ax_0.grid(True, linestyle=':', which='both')

ax_0.set_xlim(t_min, t_max)
ax_0.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
#ax_0.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))

ax_1.plot(S_par_128_window.time, S_par_128_window.data*1E3, c='k', linewidth=1)
#ax_1.scatter(S_par_128_window.time, S_par_128_window.data*1E3, c='k', s=0.1)
ax_1.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
ax_1.set_ylabel(S_par_label + '\n' + r'[$\mathrm{mW/m^{2}}$]')
ax_1.minorticks_on()
ax_1.grid(True, linestyle=':', which='both')

ax_2.plot(v_sys_window.time, v_sys_window.data*1E-3, c='k', linewidth=1)
ax_2.axhline(0, c='gray', linewidth=2, linestyle='dashed', alpha=0.7)
ax_2.set_ylabel(v_sys_label + '\n' + r'[$\mathrm{km/s}$]')
ax_2.minorticks_on()
ax_2.grid(True, linestyle=':', which='both')
ax_2.set_xlabel('Time')

ax_0.set_xlim(t_min, t_max)

fig_kappa.tight_layout()

# プロットを画像として保存
kappa_base      = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}'
kappa_png_path  = os.path.join(out_dir, kappa_base + '.png')
kappa_pdf_path  = os.path.join(out_dir, kappa_base + '.pdf')
fig_kappa.savefig(kappa_png_path, dpi=200, bbox_inches='tight')
fig_kappa.savefig(kappa_pdf_path, bbox_inches='tight')
plt.close(fig_kappa)
print(f"Time series plot of kappa saved to {kappa_png_path}")

# 以下、過去

In [ ]:
#da_zeros_spin       = xr.zeros_like(joined_EBspin_fac_cwt['Espin_fac_x_cwt'][0])
#time_interp_base    = (joined_EB128_fac_cwt['E128_fac_x_cwt'][0]).time
#da_zeros_spin       = da_zeros_spin.interp(time=time_interp_base).sortby('freq')
#
#da_E_spin_fac_x_interp  = (joined_EBspin_fac_cwt['Espin_fac_x_cwt'][0]).interp(time=time_interp_base).sortby('freq')
#da_E_spin_fac_y_interp  = (joined_EBspin_fac_cwt['Espin_fac_y_cwt'][0]).interp(time=time_interp_base).sortby('freq')
#da_B_spin_fac_x_interp  = (joined_EBspin_fac_cwt['Bspin_fac_x_cwt'][0]).interp(time=time_interp_base).sortby('freq')
#da_B_spin_fac_y_interp  = (joined_EBspin_fac_cwt['Bspin_fac_y_cwt'][0]).interp(time=time_interp_base).sortby('freq')
#
#ds_EBspin_fac_cwt_toroidal  = xr.Dataset({
#    'Espin_fac_x_cwt':  da_E_spin_fac_x_interp,
#    'Espin_fac_y_cwt':  da_zeros_spin,
#    'Espin_fac_z_cwt':  da_zeros_spin,
#    'Bspin_fac_x_cwt':  da_zeros_spin,
#    'Bspin_fac_y_cwt':  da_B_spin_fac_y_interp,
#    'Bspin_fac_z_cwt':  da_zeros_spin,
#})
#
#ds_EBspin_fac_cwt_poloidal  = xr.Dataset({
#    'Espin_fac_x_cwt':  da_zeros_spin,
#    'Espin_fac_y_cwt':  da_E_spin_fac_y_interp,
#    'Espin_fac_z_cwt':  da_zeros_spin,
#    'Bspin_fac_x_cwt':  da_B_spin_fac_x_interp,
#    'Bspin_fac_y_cwt':  da_zeros_spin,
#    'Bspin_fac_z_cwt':  da_zeros_spin,
#})
#
#ds_EBspin_fac_cwt_perp      = xr.Dataset({
#    'Espin_fac_x_cwt':  da_E_spin_fac_x_interp,
#    'Espin_fac_y_cwt':  da_E_spin_fac_y_interp,
#    'Espin_fac_z_cwt':  da_zeros_spin,
#    'Bspin_fac_x_cwt':  da_B_spin_fac_x_interp,
#    'Bspin_fac_y_cwt':  da_B_spin_fac_y_interp,
#    'Bspin_fac_z_cwt':  da_zeros_spin,
#})
#
#print(ds_EBspin_fac_cwt_toroidal)
#print('')
#print(ds_EBspin_fac_cwt_poloidal)
#print('')
#print(ds_EBspin_fac_cwt_perp)

In [ ]:
#da_zeros_128   = xr.zeros_like(joined_EB128_fac_cwt['E128_fac_x_cwt'][0]).sortby('freq')
#
#da_E_128_fac_x_interp  = (joined_EB128_fac_cwt['E128_fac_x_cwt'][0]).sortby('freq')
#da_E_128_fac_y_interp  = (joined_EB128_fac_cwt['E128_fac_y_cwt'][0]).sortby('freq')
#da_B_128_fac_x_interp  = (joined_EB128_fac_cwt['B128_fac_x_cwt'][0]).sortby('freq')
#da_B_128_fac_y_interp  = (joined_EB128_fac_cwt['B128_fac_y_cwt'][0]).sortby('freq')
#
#ds_EB128_fac_cwt_toroidal  = xr.Dataset({
#    'E128_fac_x_cwt':  da_E_128_fac_x_interp,
#    'E128_fac_y_cwt':  da_zeros_128,
#    'E128_fac_z_cwt':  da_zeros_128,
#    'B128_fac_x_cwt':  da_zeros_128,
#    'B128_fac_y_cwt':  da_B_128_fac_y_interp,
#    'B128_fac_z_cwt':  da_zeros_128,
#})
#
#ds_EB128_fac_cwt_poloidal  = xr.Dataset({
#    'E128_fac_x_cwt':  da_zeros_128,
#    'E128_fac_y_cwt':  da_E_128_fac_y_interp,
#    'E128_fac_z_cwt':  da_zeros_128,
#    'B128_fac_x_cwt':  da_B_128_fac_x_interp,
#    'B128_fac_y_cwt':  da_zeros_128,
#    'B128_fac_z_cwt':  da_zeros_128,
#})
#
#ds_EB128_fac_cwt_perp      = xr.Dataset({
#    'E128_fac_x_cwt':  da_E_128_fac_x_interp,
#    'E128_fac_y_cwt':  da_E_128_fac_y_interp,
#    'E128_fac_z_cwt':  da_zeros_128,
#    'B128_fac_x_cwt':  da_B_128_fac_x_interp,
#    'B128_fac_y_cwt':  da_B_128_fac_y_interp,
#    'B128_fac_z_cwt':  da_zeros_128,
#})
#
#print(ds_EB128_fac_cwt_toroidal)
#print('')
#print(ds_EB128_fac_cwt_poloidal)
#print('')
#print(ds_EB128_fac_cwt_perp)

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#import module_handmade.psd_plotter_themis_xarray as psdptx
#import importlib
#importlib.reload(psdptx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction           = 'perp'
#dsets_spin_clean    = [ds_EBspin_fac_cwt_perp]
#dsets_128           = [ds_EB128_fac_cwt_perp]
#ds_velocity_ms      = ds_velocity_ms_perp
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 1
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdptx.build_data_dict_xr(
#    dsets_spin_clean, dsets_128, ds_velocity_ms, ds_parameter,
#    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
#    if fig is None:
#        return
#    try:
#        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#import module_handmade.psd_plotter_themis_xarray as psdptx
#import importlib
#importlib.reload(psdptx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction           = 'poloidal'
#dsets_spin_clean    = [ds_EBspin_fac_cwt_poloidal]
#dsets_128           = [ds_EB128_fac_cwt_poloidal]
#ds_velocity_ms      = ds_velocity_ms_poloidal
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 1
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdptx.build_data_dict_xr(
#    dsets_spin_clean, dsets_128, ds_velocity_ms, ds_parameter,
#    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
#    if fig is None:
#        return
#    try:
#        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#import module_handmade.psd_plotter_themis_xarray as psdptx
#import importlib
#importlib.reload(psdptx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 12
#
#direction           = 'toroidal'
#dsets_spin_clean    = [ds_EBspin_fac_cwt_toroidal]
#dsets_128           = [ds_EB128_fac_cwt_toroidal]
#ds_velocity_ms      = ds_velocity_ms_toroidal
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 1
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdptx.build_data_dict_xr(
#    dsets_spin_clean, dsets_128, ds_velocity_ms, ds_parameter,
#    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
#    if fig is None:
#        return
#    try:
#        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#
#time_range = ['2022-09-01T22:25:00', '2022-09-01T23:15:00']
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#import module_handmade.psd_plotter_themis_xarray as psdptx
#import importlib
#importlib.reload(psdptx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 15
#
#direction           = 'perp'
#dsets_spin          = [ds_EBspin_fac_cwt_perp]
#dsets_128           = [ds_EB128_fac_cwt_perp]
#ds_velocity_ms      = ds_velocity_ms_perp
#S_para_select       = S_para
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 1
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdptx.build_data_dict_xr(
#    dsets_spin, dsets_128, ds_velocity_ms, ds_parameter,
#    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30, fit_range=(3, 30))
#    if fig is None:
#        return
#    try:
#        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#        return fit_results
#
##time_range = ['2022-09-01T22:30:45', '2022-09-01T22:32:45']
##time_range = ['2022-09-01T22:29:00', '2022-09-01T22:34:00']
##time_range = ['2022-09-01T22:48:30', '2022-09-01T22:51:30']
#time_range = ['2022-09-01T23:05:45', '2022-09-01T23:08:15']
#
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para_select.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    #ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#    #            fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    ax_0.plot(df_results.index, df_results['kappa_B'], marker='.', c='blue', label=r'$\kappa_{\mathrm{B}}$', lw=2)
#    ax_0.axhline(7./3., lw=2, linestyle='dotted', c='blue', label=r'$\kappa_{\mathrm{B}}$ in KAW cascade')
#    
#    # kappa_Eのプロット（エラーバー付き）
#    #ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#    #            fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    ax_0.plot(df_results.index, df_results['kappa_E'], marker='.', c='orange', label=r'$\kappa_{\mathrm{E}}$', lw=2)
#    ax_0.axhline(1./3., lw=2, linestyle='dotted', c='orange', label=r'$\kappa_{\mathrm{E}}$ in KAW cascade')
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)' + f' ({direction})')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend(loc='best', ncol=4, fontsize=12)
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#import module_handmade.psd_plotter_themis_xarray as psdptx
#import importlib
#importlib.reload(psdptx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 15
#
#direction           = 'poloidal'
#dsets_spin          = [ds_EBspin_fac_cwt_poloidal]
#dsets_128           = [ds_EB128_fac_cwt_poloidal]
#ds_velocity_ms      = ds_velocity_ms_poloidal
#S_para_select       = S_para_poloidal
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 1
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdptx.build_data_dict_xr(
#    dsets_spin, dsets_128, ds_velocity_ms, ds_parameter,
#    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
#    if fig is None:
#        return
#    try:
#        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#        return fit_results
#
##time_range = ['2022-09-01T22:29:00', '2022-09-01T22:34:00']
##time_range = ['2022-09-01T22:47:30', '2022-09-01T22:52:30']
#time_range = ['2022-09-01T23:04:00', '2022-09-01T23:09:00']
#
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para_select.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)' + f'({direction})')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")

In [ ]:
#import os
#import numpy as np
#import pandas as pd
#from joblib import Parallel, delayed
#import matplotlib as mpl
#import matplotlib.pyplot as plt
#
## --- モジュール読み込み ---
#import sys
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#import module_handmade.psd_plotter_themis_xarray as psdptx
#import importlib
#importlib.reload(psdptx)
#
#mpl.rcdefaults()
#mpl.rcParams['font.size'] = 15
#
#direction           = 'toroidal'
#dsets_spin          = [ds_EBspin_fac_cwt_toroidal]
#dsets_128           = [ds_EB128_fac_cwt_toroidal]
#ds_velocity_ms      = ds_velocity_ms_toroidal
#S_para_select       = S_para_toroidal
#
## ------------------------------------------------------------
## 0. 出力フォルダ
## ------------------------------------------------------------
#dt_time = 1
#out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/22-24_CWT_{dt_time}sec_k_rhoi_{direction}'
#os.makedirs(out_dir, exist_ok=True)
#
## ------------------------------------------------------------
## 1. データ準備（ここは一度だけ）
## ------------------------------------------------------------
#
#data_dict = psdptx.build_data_dict_xr(
#    dsets_spin, dsets_128, ds_velocity_ms, ds_parameter,
#    f_split=0.3648/2., cutoff_freq=[1/100, 0.7/3, 5/3, 64]
#)
#
#if data_dict is None:
#    print("データ準備に失敗。終了。")
#    raise SystemExit
#
## ------------------------------------------------------------
## 2. X秒ごとにプロットして保存（並列）
## ------------------------------------------------------------
#def process_and_save_plot(t_start, dd, dt, out_dir):
#    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
#    if fig is None:
#        return
#    try:
#        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
#        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
#    finally:
#        plt.close(fig)
#        return fit_results
#
##time_range = ['2022-09-01T22:29:00', '2022-09-01T22:34:00']
##time_range = ['2022-09-01T22:47:30', '2022-09-01T22:52:30']
#time_range = ['2022-09-01T23:04:00', '2022-09-01T23:09:00']
#
#t_min, t_max = pd.to_datetime(time_range)
#time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')
#
#print(f"Processing {len(time_steps)} plots in parallel...")
#
#results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
#    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
#)
#
#print('Finished saving all plots!')
#
#print("\n--- Saving and plotting fitting results ---")
#
## Noneが含まれる可能性を考慮してフィルタリング
#results_list = [r for r in results_list if r is not None]
#
#if results_list:
#    # リストからDataFrameを作成
#    df_results = pd.DataFrame(results_list)
#    df_results = df_results.set_index('time').sort_index()
#
#    # CSVファイルとして保存
#    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
#    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
#    kappa_csv_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.csv'
#    csv_path = os.path.join(out_dir, kappa_csv_filename)
#    df_results.to_csv(csv_path)
#    print(f"Fitting results saved to {csv_path}")
#
#    S_para_window   = S_para_select.sel(time=slice(t_min, t_max))
#
#    # --- 時間変化をプロット ---
#    fig_kappa = plt.figure(figsize=(10, 6))
#    gs = fig_kappa.add_gridspec(3, 1)
#    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
#    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)
#
#    ax_0.tick_params(axis='x', which='both', labelbottom=False)
#    
#    # kappa_Bのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
#                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    # kappa_Eのプロット（エラーバー付き）
#    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
#                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
#    
#    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)' + f'({direction})')
#    ax_0.set_ylabel(r'Spectral index $\kappa$')
#    ax_0.legend()
#    ax_0.minorticks_on()
#    ax_0.grid(True, linestyle=':', which='both')
#
#    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
#    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#    ax_1.minorticks_on()
#    ax_1.grid(True, linestyle=':', which='both')
#    ax_1.set_xlabel('Time')
#
#    ax_0.set_xlim(t_min, t_max)
#
#    fig_kappa.tight_layout()
#
#    # プロットを画像として保存
#    kappa_plot_filename = f'kappa_timeseries_{direction}_{time_str_start}_to_{time_str_end}.png'
#    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
#    fig_kappa.savefig(kappa_plot_path, dpi=200)
#    plt.close(fig_kappa)
#    print(f"Time series plot of kappa saved to {kappa_plot_path}")
#
#else:
#    print("No fitting results to save or plot.")